# Evaluacion Transversal — Predicción de abandono de clientes en ventas y marketing


**Integrantes:**

- Sebastian Lagos
- Oscar Oreste

**Contexto**

Una empresa del área de ventas y marketing dispone de información demográfica,
comercial y de experiencia de sus clientes. El objetivo del proyecto es analizar
estos datos y desarrollar un modelo de clasificación que permita identificar
clientes con riesgo de abandono.

La unidad de análisis corresponde a cada cliente y la variable objetivo es
`churn`, donde:

- `0`: el cliente no abandonó.
- `1`: el cliente abandonó.

El resultado puede apoyar decisiones de retención, marketing y experiencia del
cliente.

## Objetivos del proyecto

### Objetivo general

Desarrollar una solución reproducible de ciencia de datos para identificar
clientes con riesgo de abandono mediante técnicas de limpieza, integración de
datos, análisis exploratorio, clasificación y visualización interactiva.

### Objetivos específicos

1. Auditar la estructura y calidad del dataset.
2. Aplicar reglas de limpieza reproducibles sin modificar el archivo original.
3. Integrar indicadores externos mediante una API.
4. Analizar las características asociadas al abandono.
5. Comparar modelos de clasificación mediante accuracy.
6. Interpretar los resultados utilizando una matriz de confusión.
7. Construir un dashboard interactivo con Plotly 

## 1. Importación de librerías

Se importan las librerías principales que se utilizarán para cargar, revisar y
trabajar con el dataset.

Las librerías de visualización, API y Machine Learning se incorporarán en las
secciones correspondientes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Carga y procedencia del dataset

El dataset base corresponde a **Sales and Marketing DataSet**, publicado por
**Bhasker Paul** en Kaggle:

https://www.kaggle.com/datasets/bhaskerpaul/sales-and-marketing-dataset

El nombre original del archivo es `Sales - Marketing customer dataset.csv`.
Dentro del repositorio fue renombrado como
`data/raw/sales_marketing_customer_dataset.csv` para mantener una convención
uniforme. El cambio de nombre no modifica su contenido ni procedencia.

El archivo ubicado en `data/raw` se considera inmutable. Se utilizan dos rutas
posibles para permitir que el notebook pueda ejecutarse desde la raíz del
repositorio o desde la carpeta `notebook`.

In [2]:
rutas_posibles = [
    Path("data/raw/sales_marketing_customer_dataset.csv"),
    Path("../data/raw/sales_marketing_customer_dataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró sales_marketing_customer_dataset.csv en data/raw."
    )

df = pd.read_csv(ruta_dataset)

print("Dataset cargado correctamente.")
print("Ruta utilizada:", ruta_dataset)

Dataset cargado correctamente.
Ruta utilizada: ..\data\raw\sales_marketing_customer_dataset.csv


## 3. Vista inicial del dataset

Se muestran las primeras filas para conocer la estructura general y observar
ejemplos de los datos disponibles.

In [3]:
df.head()

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,is_premium_user,total_visits,avg_session_time,pages_per_session,email_open_rate,email_click_rate,total_spent,avg_order_value,discount_used,coupon_code,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.00,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,1,7,13.90,5.42,0.67,0.26,559.52,65.25,0,NEW20,0,0,3,UPI,3.00,10,27.56,915.31,14,0
1,10002,NaN,35.00,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,0,19,5.11,5.35,0.70,0.37,356.49,48.47,1,NEW20,5,0,3,BKash,3.00,7,15.15,"2,079.96",11,0
2,10003,Female,27.00,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,1,18,9.74,3.59,0.47,0.44,689.33,77.82,0,NaN,1,0,2,UPI,5.00,6,13.51,"1,379.15",9,0
3,10004,Female,36.00,India,Mumbai,2024-01-28 00:00:00,2023-05-20 00:00:00,Facebook Ads,Tablet,Annual,1,16,9.64,2.95,0.58,0.37,445.43,71.71,0,NaN,0,0,2,PayPal,4.00,6,25.65,774.65,7,0
4,10005,Male,29.00,USA,Hamburg,2023-07-21 00:00:00,2024-04-07 00:00:00,Referral,Mobile,Monthly,0,12,7.79,2.41,0.05,0.16,686.29,44.99,1,NaN,2,1,4,BKash,3.00,1,12.39,87.68,11,0


## 4. Inspección inicial del dataset

Se revisan las dimensiones, los nombres de las columnas y los tipos de datos.

Esta inspección permite comprender la estructura antes de aplicar cualquier
limpieza o transformación.

In [4]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

Dimensión del dataset (filas, columnas):
(15000, 30)

Columnas del dataset:
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type', 'is_premium_user', 'total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'payment_method', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 15000 non-null  int64  
 1   gender                      14262 non-null  str    
 2   age                         13800 non-null

## 5. Resumen estadístico inicial

Se calculan estadísticas descriptivas para las variables numéricas.

Esto permite observar sus valores mínimos, máximos, promedios y posibles
diferencias de escala.

In [5]:
print("Resumen estadístico de las columnas numéricas:")
display(df.describe().T)

Resumen estadístico de las columnas numéricas:


,count,mean,std,min,25%,50%,75%,max
customer_id,"15,000.00","17,500.50","4,330.27","10,001.00","13,750.75","17,500.50","21,250.25","25,000.00"
age,"13,800.00",35.20,10.33,-4.00,28.00,35.00,42.00,95.00
is_premium_user,"15,000.00",0.30,0.46,0.00,0.00,0.00,1.00,1.00
total_visits,"15,000.00",15.00,3.89,3.00,12.00,15.00,18.00,31.00
avg_session_time,"15,000.00",8.02,2.99,0.01,5.97,7.99,10.06,19.12
pages_per_session,"15,000.00",4.00,1.48,0.01,2.99,4.00,5.01,10.84
email_open_rate,"15,000.00",0.50,0.29,0.00,0.24,0.50,0.75,1.00
email_click_rate,"15,000.00",0.25,0.14,0.00,0.13,0.25,0.38,0.50
total_spent,"13,950.00",524.36,467.05,0.27,300.43,498.84,702.40,"15,910.43"
avg_order_value,"15,000.00",60.08,24.75,0.07,43.03,60.11,76.89,154.55


## 6. Revisión del identificador de clientes

La columna `customer_id` identifica a cada cliente. Se comprueba su cantidad de
valores únicos y si existen identificadores repetidos.

In [6]:
print("Cantidad de registros:", len(df))
print("Clientes únicos:", df["customer_id"].nunique())
print(
    "Identificadores duplicados:",
    df["customer_id"].duplicated().sum()
)

Cantidad de registros: 15000
Clientes únicos: 15000
Identificadores duplicados: 0


## 7. Distribución de la variable objetivo

La variable `churn` indica si un cliente abandonó o no la empresa:

- `0`: no abandonó;
- `1`: abandonó.

Se revisa la cantidad y el porcentaje de clientes pertenecientes a cada clase.

In [7]:
distribucion_churn = df["churn"].value_counts().sort_index()

porcentaje_churn = (
    df["churn"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

resumen_churn = pd.DataFrame({
    "Cantidad": distribucion_churn,
    "Porcentaje": porcentaje_churn
})

display(resumen_churn)

,Cantidad,Porcentaje
churn,,
0,12702,84.68
1,2298,15.32


## 8. Baseline de accuracy

Como existe una clase mayoritaria, se calcula el resultado que obtendría una
regla que predijera siempre la clase más frecuente.

Este valor se utilizará posteriormente como referencia para evaluar los modelos
de clasificación.

In [8]:
clase_mayoritaria = df["churn"].mode()[0]

baseline_accuracy = (
    df["churn"] == clase_mayoritaria
).mean()

print("Clase mayoritaria:", clase_mayoritaria)
print(f"Baseline de accuracy: {baseline_accuracy:.2%}")

Clase mayoritaria: 0
Baseline de accuracy: 84.68%


### Interpretación inicial

El dataset contiene 15.000 registros y cada fila representa un cliente.

La clase `0`, correspondiente a clientes que no abandonaron la empresa, es
considerablemente más frecuente que la clase `1`.

Por este motivo, una accuracy cercana al 84,68 % no necesariamente representa
un buen modelo, ya que ese resultado puede alcanzarse prediciendo siempre la
clase mayoritaria.

Los modelos deberán superar o aportar información adicional respecto de este
baseline y sus resultados serán complementados con una matriz de confusión.

## 9. Revisión de calidad de los datos

Antes de limpiar el dataset se revisan los principales problemas de calidad:

- registros duplicados;
- valores nulos;
- categorías inconsistentes;
- edades inválidas;
- fechas incoherentes;
- relación entre país y ciudad;
- valores extremos.

En esta etapa solo se detectan y cuantifican los problemas. No se modifica el
DataFrame original.

## 10. Detección de duplicados

Se revisan primero los registros completamente duplicados.

También se comprueba si existen clientes con exactamente los mismos datos,
ignorando el identificador `customer_id`.

In [9]:
duplicados_exactos = df.duplicated().sum()

duplicados_sin_id = (
    df.drop(columns="customer_id")
    .duplicated()
    .sum()
)

print("Duplicados exactos:", duplicados_exactos)
print("Duplicados ignorando customer_id:", duplicados_sin_id)

Duplicados exactos: 0
Duplicados ignorando customer_id: 0


## 11. Detección de valores nulos

Se calcula la cantidad y el porcentaje de valores faltantes por columna.

El porcentaje permite evaluar si conviene imputar, conservar una categoría
explícita o excluir una variable del análisis.

In [10]:
resumen_nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje": (df.isnull().mean() * 100).round(2)
})

resumen_nulos = (
    resumen_nulos[resumen_nulos["Cantidad"] > 0]
    .sort_values("Cantidad", ascending=False)
)

display(resumen_nulos)

,Cantidad,Porcentaje
coupon_code,6133,40.89
age,1200,8.00
total_spent,1050,7.00
gender,738,4.92
satisfaction_score,702,4.68


### Interpretación de los valores nulos

Los valores faltantes no deben tratarse todos de la misma manera.

- En `coupon_code`, la ausencia no permite concluir si el cliente no utilizó un
  cupón o si el código simplemente no fue registrado. Por ello se conservará
  una categoría explícita de falta de registro y se creará una bandera de
  presencia del código.
- `gender` se conservará con una categoría explícita de información no
  registrada, sin inferir el género.
- `age`, `total_spent` y `satisfaction_score` son variables numéricas y su
  imputación se realizará posteriormente dentro del pipeline de Machine
  Learning.

No se utilizará el promedio o la mediana sobre el dataset completo, porque eso
podría introducir información del conjunto de prueba en el preprocesamiento.

## 12. Revisión de variables categóricas

Se inspeccionan las variables de texto con pocas categorías para detectar
diferencias de escritura, espacios adicionales o categorías vacías.

Las columnas de fechas se excluyen de esta revisión porque serán analizadas por
separado.

In [11]:
columnas_fecha = ["signup_date", "last_purchase_date"]

columnas_categoricas = [
    columna
    for columna in df.columns
    if columna not in columnas_fecha
    and pd.api.types.is_string_dtype(df[columna])
    and df[columna].nunique(dropna=True) <= 15
]

for columna in columnas_categoricas:
    print(f"\nVariable: {columna}")
    display(
        df[columna]
        .value_counts(dropna=False)
        .to_frame("Cantidad")
    )


Variable: gender


,Cantidad
gender,
Male,6844
Female,6686
NaN,738
Other,732



Variable: country


,Cantidad
country,
Germany,3072
India,3014
Bangladesh,2984
USA,2975
UK,2955



Variable: city


...

,Cantidad
subscription_type,
Monthly,7666
Annual,7334



Variable: coupon_code


,Cantidad
coupon_code,
NaN,6133
REF10,2995
SALE15,2986
NEW20,2886



Variable: payment_method


,Cantidad
payment_method,
UPI,3105
PayPal,3005
SEPA,2986
BKash,2971
Card,2933


## 13. Revisión de la edad

Se revisan los valores mínimos y máximos de `age`.

Las edades menores o iguales a cero se consideran claramente inválidas. Los
clientes menores de 18 o mayores de 80 se informan como observaciones, pero no
se eliminan automáticamente porque podrían ser registros reales.

In [12]:
resumen_edad = pd.DataFrame({
    "Indicador": [
        "Edad mínima",
        "Edad máxima",
        "Edades negativas",
        "Edad igual a cero",
        "Menores de 18 años",
        "Mayores de 80 años"
    ],
    "Cantidad o valor": [
        df["age"].min(),
        df["age"].max(),
        (df["age"] < 0).sum(),
        (df["age"] == 0).sum(),
        (df["age"] < 18).sum(),
        (df["age"] > 80).sum()
    ]
})

display(resumen_edad)

,Indicador,Cantidad o valor
0,Edad mínima,-4.00
1,Edad máxima,95.00
2,Edades negativas,3.00
3,Edad igual a cero,1.00
4,Menores de 18 años,516.00
5,Mayores de 80 años,30.00


### Interpretación de la edad

Solo las edades menores o iguales a cero se consideran inválidas sin requerir
una regla de negocio adicional.

Durante la limpieza, estos valores se convertirán en nulos. La imputación de la
edad se realizará posteriormente dentro del pipeline y se ajustará únicamente
con los datos de entrenamiento.

## 14. Revisión de fechas

Las columnas de fechas se convierten temporalmente para comprobar:

- formatos inválidos;
- fecha mínima y máxima;
- clientes cuya última compra aparece antes de la fecha de registro.

En esta etapa no se reemplazan ni se sobrescriben las columnas originales.

In [13]:
signup_date_temp = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

last_purchase_date_temp = pd.to_datetime(
    df["last_purchase_date"],
    errors="coerce"
)

signup_invalidas = (
    df["signup_date"].notna()
    & signup_date_temp.isna()
).sum()

last_purchase_invalidas = (
    df["last_purchase_date"].notna()
    & last_purchase_date_temp.isna()
).sum()

secuencia_fecha_invalida = (
    signup_date_temp.notna()
    & last_purchase_date_temp.notna()
    & (last_purchase_date_temp < signup_date_temp)
)

print("Fechas de registro inválidas:", signup_invalidas)
print("Fechas de última compra inválidas:", last_purchase_invalidas)

print("\nRango de signup_date:")
print(signup_date_temp.min(), "a", signup_date_temp.max())

print("\nRango de last_purchase_date:")
print(last_purchase_date_temp.min(), "a", last_purchase_date_temp.max())

print(
    "\nÚltima compra anterior al registro:",
    secuencia_fecha_invalida.sum()
)

print(
    "Porcentaje de inconsistencias:",
    f"{secuencia_fecha_invalida.mean() * 100:.2f}%"
)

Fechas de registro inválidas: 0
Fechas de última compra inválidas: 0

Rango de signup_date:
2022-01-01 00:00:00 a 2024-09-26 00:00:00

Rango de last_purchase_date:
2023-01-01 00:00:00 a 2025-03-10 00:00:00

Última compra anterior al registro: 3762
Porcentaje de inconsistencias: 25.08%


### Interpretación de las fechas

Las fechas presentan un formato válido, pero existe una cantidad importante de
registros donde la última compra aparece antes de la fecha de inscripción.

Estas fechas no se corregirán inventando valores ni utilizando una fecha
promedio o mediana.

En la limpieza se creará una bandera de inconsistencia y la antigüedad solo se
calculará cuando el orden temporal sea válido.

## 15. Revisión de consistencia entre país y ciudad

Se revisa si una misma ciudad aparece asociada a diferentes países dentro del
dataset.

Para realizar una comprobación interna, se utiliza como referencia el país más
frecuente asociado a cada ciudad.

Esta regla permite detectar asociaciones inusuales, pero no constituye una
validación geográfica externa definitiva.

In [14]:
ubicaciones = (
    df[["country", "city"]]
    .dropna()
    .copy()
)

pais_frecuente_por_ciudad = (
    ubicaciones
    .groupby("city")["country"]
    .agg(lambda valores: valores.mode().iloc[0])
)

ubicaciones["pais_referencia_interno"] = (
    ubicaciones["city"]
    .map(pais_frecuente_por_ciudad)
)

ubicaciones["inconsistencia_interna"] = (
    ubicaciones["country"]
    != ubicaciones["pais_referencia_interno"]
)

cantidad_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .sum()
)

porcentaje_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .mean()
    * 100
)

print(
    "Combinaciones inusuales según la relación interna:",
    cantidad_inconsistencias
)

print(
    "Porcentaje:",
    f"{porcentaje_inconsistencias:.2f}%"
)


display(
    ubicaciones[
        ubicaciones["inconsistencia_interna"]
    ].head(10)
)

Combinaciones inusuales según la relación interna:

 11818
Porcentaje: 78.79%


,country,city,pais_referencia_interno,inconsistencia_interna
2,Germany,London,USA,True
3,India,Mumbai,Germany,True
4,USA,Hamburg,UK,True
6,UK,New York,India,True
7,Germany,Berlin,India,True
9,Germany,Hamburg,UK,True
10,Germany,Delhi,USA,True
11,Bangladesh,New York,India,True
12,Germany,Hamburg,UK,True
13,India,Mumbai,Germany,True


### Interpretación de ubicación

La asociación dominante entre ciudad y país muestra una cantidad elevada de
combinaciones internas inusuales.

No se corregirá la ciudad mediante suposiciones. En el modelo inicial se
priorizará `country` y se evaluará excluir `city` debido a su inconsistencia.

La validación externa mediante la API se realizará por país, no por ciudad.

## 16. Revisión de valores extremos en total_spent

Se utiliza el rango intercuartílico para identificar valores extremos en
`total_spent`.

El método IQR define como posibles outliers los valores menores a:

Q1 - 1.5 × IQR

o mayores a:

Q3 + 1.5 × IQR

En esta etapa los registros solo se identifican. No se eliminan ni se truncan.

In [15]:
total_spent_sin_nulos = df["total_spent"].dropna()

q1_total_spent = total_spent_sin_nulos.quantile(0.25)
q3_total_spent = total_spent_sin_nulos.quantile(0.75)

iqr_total_spent = q3_total_spent - q1_total_spent

limite_inferior_total_spent = (
    q1_total_spent - 1.5 * iqr_total_spent
)

limite_superior_total_spent = (
    q3_total_spent + 1.5 * iqr_total_spent
)

filtro_outliers_total_spent = (
    df["total_spent"].notna()
    & (
        (df["total_spent"] < limite_inferior_total_spent)
        | (df["total_spent"] > limite_superior_total_spent)
    )
)

cantidad_outliers = filtro_outliers_total_spent.sum()

print("Mediana:", round(total_spent_sin_nulos.median(), 2))
print("Percentil 99:", round(total_spent_sin_nulos.quantile(0.99), 2))
print("Valor máximo:", round(total_spent_sin_nulos.max(), 2))

print("\nLímite inferior IQR:", round(limite_inferior_total_spent, 2))
print("Límite superior IQR:", round(limite_superior_total_spent, 2))

print("\nCantidad de valores extremos:", cantidad_outliers)
print(
    "Porcentaje:",
    f"{cantidad_outliers / len(df) * 100:.2f}%"
)

Mediana: 498.84
Percentil 99: 1216.01
Valor máximo: 15910.43

Límite inferior IQR: -302.51
Límite superior IQR: 1305.34

Cantidad de valores extremos: 78
Porcentaje: 0.52%


### Interpretación de los valores extremos

Los valores extremos de gasto no se eliminarán automáticamente, porque podrían
corresponder a clientes reales de alto valor.

En fases posteriores se evaluarán tres alternativas:

1. conservar la variable original;
2. crear una transformación logarítmica;
3. aplicar truncamiento únicamente si resulta necesario.

Cualquier límite utilizado para el modelo deberá calcularse con los datos de
entrenamiento y no con el dataset completo.

## 17. Revisión de variables constantes

Una variable constante contiene el mismo valor en todos los registros y no
aporta información para diferenciar clientes.

Se revisa si existen columnas con uno o ningún valor único.

In [16]:
cardinalidad = df.nunique(dropna=False)

variables_constantes = cardinalidad[
    cardinalidad <= 1
]

if variables_constantes.empty:
    print("No se detectaron variables constantes.")
else:
    print("Variables constantes detectadas:")
    display(variables_constantes.to_frame("Valores únicos"))

No se detectaron variables constantes.


## Conclusión de la auditoría de calidad

La auditoría permitió identificar problemas reales que justifican una etapa de
limpieza y transformación:

- existen valores nulos en variables numéricas y categóricas;
- se detectan edades menores o iguales a cero;
- las fechas tienen formato válido, pero existen secuencias temporales
  incoherentes;
- la relación entre ciudad y país presenta inconsistencias;
- `total_spent` contiene valores extremos;
- no se detectaron duplicados relevantes.

Durante esta etapa no se modificaron los datos.

En la siguiente fase se aplicarán reglas simples y justificadas:

- copia del DataFrame;
- conversión de edades inválidas en nulos;
- tratamiento explícito de categorías faltantes;
- conversión definitiva de fechas;
- creación de banderas de inconsistencia;
- creación de variables temporales válidas;
- preparación de las variables numéricas para su imputación posterior dentro
  del pipeline.

## 18. Limpieza e ingeniería inicial

A partir de la auditoría se crea una copia de trabajo llamada `df_clean`.

El DataFrame original `df` no se sobrescribe y el archivo ubicado en
`data/raw` permanece intacto. En esta fase se aplican únicamente reglas
determinísticas y justificadas; no se realiza imputación estadística, modelado,
integración de API ni análisis exploratorio completo.

In [17]:
df_clean = df.copy()

filas_antes = len(df_clean)
duplicados_antes = int(df_clean.duplicated().sum())

if duplicados_antes > 0:
    df_clean = df_clean.drop_duplicates().copy()

filas_despues_duplicados = len(df_clean)
duplicados_eliminados = filas_antes - filas_despues_duplicados

print("Filas antes de revisar duplicados:", filas_antes)
print("Duplicados exactos detectados:", duplicados_antes)
print("Duplicados exactos eliminados:", duplicados_eliminados)
print("Filas después de revisar duplicados:", filas_despues_duplicados)

Filas antes de revisar duplicados: 15000
Duplicados exactos detectados: 0
Duplicados exactos eliminados: 0
Filas después de revisar duplicados: 15000


### 18.1 Normalización de variables categóricas

Se eliminan espacios al inicio y al final de las variables de texto y las
cadenas vacías se convierten en valores faltantes.

La auditoría no mostró diferencias de escritura que justificaran cambiar
mayúsculas o minúsculas, por lo que no se recodifican categorías válidas.

En `gender` y `coupon_code` se utiliza la categoría `Sin registro` para
distinguir la ausencia de información. Esto no significa que se haya inferido
el género ni que un cliente necesariamente no haya utilizado un cupón.

In [18]:
columnas_texto_limpieza = [
    "gender",
    "country",
    "city",
    "acquisition_channel",
    "device_type",
    "subscription_type",
    "coupon_code",
    "payment_method"
]

for columna in columnas_texto_limpieza:
    df_clean[columna] = df_clean[columna].str.strip()
    df_clean[columna] = df_clean[columna].replace("", pd.NA)

gender_sin_registro_antes = int(df_clean["gender"].isna().sum())
coupon_sin_registro_antes = int(df_clean["coupon_code"].isna().sum())

df_clean["coupon_code_present_flag"] = (
    df_clean["coupon_code"].notna().astype(int)
)

df_clean["gender"] = df_clean["gender"].fillna("Sin registro")
df_clean["coupon_code"] = (
    df_clean["coupon_code"].fillna("Sin registro")
)

print("Gender sin registro identificados:", gender_sin_registro_antes)
print("Coupon code sin registro identificados:", coupon_sin_registro_antes)

print("\nCategorías finales de gender:")
display(df_clean["gender"].value_counts(dropna=False).to_frame("Cantidad"))

print("\nCategorías finales de coupon_code:")
display(df_clean["coupon_code"].value_counts(dropna=False).to_frame("Cantidad"))

Gender sin registro identificados: 738
Coupon code sin registro identificados: 6133

Categorías finales de gender:


,Cantidad
gender,
Male,6844
Female,6686
Sin registro,738
Other,732



Categorías finales de coupon_code:


,Cantidad
coupon_code,
Sin registro,6133
REF10,2995
SALE15,2986
NEW20,2886


### 18.2 Tratamiento de edades inválidas

Las edades menores o iguales a cero se consideran inválidas y se convierten en
`NaN`.

Los clientes menores de 18 años o mayores de 80 se conservan porque, sin una
regla de negocio adicional, no corresponde eliminarlos ni modificar sus
edades.

La imputación de `age` se realizará posteriormente dentro del pipeline y se
ajustará únicamente con los datos de entrenamiento.

In [19]:
edades_invalidas_antes = int((df_clean["age"] <= 0).sum())

df_clean.loc[df_clean["age"] <= 0, "age"] = np.nan

edades_invalidas_despues = int((df_clean["age"] <= 0).sum())

print("Edades menores o iguales a cero antes:", edades_invalidas_antes)
print("Edades menores o iguales a cero después:", edades_invalidas_despues)
print("Valores nulos actuales en age:", int(df_clean["age"].isna().sum()))

Edades menores o iguales a cero antes: 4
Edades menores o iguales a cero después: 0
Valores nulos actuales en age: 1204


### 18.3 Conversión de fechas y variables temporales

Las columnas `signup_date` y `last_purchase_date` se convierten de forma
definitiva a tipo fecha mediante `pd.to_datetime(..., errors="coerce")`.

Se crea `date_inconsistency_flag`:

- `1`: la última compra es anterior al registro;
- `0`: ambas fechas son válidas y el orden es correcto;
- valor faltante: no existen dos fechas válidas para evaluar la secuencia.

`tenure_days` se calcula únicamente cuando ambas fechas son válidas y la última
compra no es anterior al registro. Por ello, nunca se generan antigüedades
negativas.

In [20]:
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"],
    errors="coerce"
)

df_clean["last_purchase_date"] = pd.to_datetime(
    df_clean["last_purchase_date"],
    errors="coerce"
)

ambas_fechas_validas = (
    df_clean["signup_date"].notna()
    & df_clean["last_purchase_date"].notna()
)

df_clean["date_inconsistency_flag"] = pd.Series(
    pd.NA,
    index=df_clean.index,
    dtype="Int64"
)

df_clean.loc[
    ambas_fechas_validas,
    "date_inconsistency_flag"
] = (
    df_clean.loc[
        ambas_fechas_validas,
        "last_purchase_date"
    ]
    < df_clean.loc[
        ambas_fechas_validas,
        "signup_date"
    ]
).astype(int)

secuencia_temporal_valida = (
    ambas_fechas_validas
    & (
        df_clean["last_purchase_date"]
        >= df_clean["signup_date"]
    )
)

df_clean["tenure_days"] = np.nan

df_clean.loc[
    secuencia_temporal_valida,
    "tenure_days"
] = (
    df_clean.loc[
        secuencia_temporal_valida,
        "last_purchase_date"
    ]
    - df_clean.loc[
        secuencia_temporal_valida,
        "signup_date"
    ]
).dt.days

df_clean["signup_year"] = (
    df_clean["signup_date"].dt.year.astype("Int64")
)

df_clean["last_purchase_year"] = (
    df_clean["last_purchase_date"].dt.year.astype("Int64")
)

print("Tipo de signup_date:", df_clean["signup_date"].dtype)
print("Tipo de last_purchase_date:", df_clean["last_purchase_date"].dtype)
print(
    "Inconsistencias temporales:",
    int(df_clean["date_inconsistency_flag"].eq(1).sum())
)
print(
    "Registros sin dos fechas válidas:",
    int(df_clean["date_inconsistency_flag"].isna().sum())
)
print(
    "Antigüedades negativas:",
    int((df_clean["tenure_days"] < 0).sum())
)

Tipo de signup_date: datetime64[us]
Tipo de last_purchase_date: datetime64[us]
Inconsistencias temporales: 3762
Registros sin dos fechas válidas: 0
Antigüedades negativas: 0


### 18.4 Bandera de inconsistencia de ubicación

No se corrigen ciudades ni se utiliza un diccionario geográfico inventado.

Se replica la regla interna de la auditoría: para cada ciudad se obtiene el país
más frecuente dentro del dataset y se marca con `1` cada asociación que no
coincide con esa referencia.

Esta bandera permite conservar el problema de calidad para auditoría, pero no
demuestra que una combinación sea geográficamente incorrecta. `country` será la
variable geográfica principal y `city` quedará fuera del primer modelo.

In [21]:
ubicacion_valida = (
    df_clean["country"].notna()
    & df_clean["city"].notna()
)

pais_referencia_por_ciudad = (
    df_clean.loc[ubicacion_valida]
    .groupby("city")["country"]
    .agg(lambda valores: valores.mode().iloc[0])
)

pais_referencia_filas = (
    df_clean.loc[ubicacion_valida, "city"]
    .map(pais_referencia_por_ciudad)
)

df_clean["location_inconsistency_flag"] = pd.Series(
    pd.NA,
    index=df_clean.index,
    dtype="Int64"
)

df_clean.loc[
    ubicacion_valida,
    "location_inconsistency_flag"
] = (
    df_clean.loc[ubicacion_valida, "country"]
    != pais_referencia_filas
).astype(int)

print(
    "Inconsistencias internas de ubicación:",
    int(df_clean["location_inconsistency_flag"].eq(1).sum())
)
print(
    "Registros sin ubicación evaluable:",
    int(df_clean["location_inconsistency_flag"].isna().sum())
)

Inconsistencias internas de ubicación:

 11818


Registros sin ubicación evaluable: 0


### 18.5 Transformación de `total_spent`

Los valores extremos de gasto se conservan porque pueden representar clientes
reales de alto valor.

Como todos los valores observados de `total_spent` son no negativos, se crea
`total_spent_log` mediante `np.log1p`. Esta transformación reduce la asimetría
sin eliminar ni truncar clientes.

Los valores faltantes se mantienen como `NaN`. Cualquier imputación o
truncamiento futuro deberá ajustarse únicamente con el conjunto de
entrenamiento.

In [22]:
valores_negativos_total_spent = int(
    (df_clean["total_spent"].dropna() < 0).sum()
)

if valores_negativos_total_spent > 0:
    raise ValueError(
        "total_spent contiene valores negativos y no permite aplicar log1p "
        "sin una revisión adicional."
    )

df_clean["total_spent_log"] = np.log1p(
    df_clean["total_spent"]
)

print(
    "Valores negativos en total_spent:",
    valores_negativos_total_spent
)
print(
    "Nulos conservados en total_spent:",
    int(df_clean["total_spent"].isna().sum())
)
print(
    "Nulos conservados en total_spent_log:",
    int(df_clean["total_spent_log"].isna().sum())
)
print(
    "Máximo de total_spent:",
    round(df_clean["total_spent"].max(), 2)
)
print(
    "Máximo de total_spent_log:",
    round(df_clean["total_spent_log"].max(), 2)
)

Valores negativos en total_spent: 0
Nulos conservados en total_spent: 1050


Nulos conservados en total_spent_log: 1050
Máximo de total_spent: 15910.43
Máximo de total_spent_log: 9.67


### 18.6 Validación posterior a la limpieza

Se comprueba que:

- el número de filas solo cambie si existían duplicados exactos;
- el DataFrame original permanezca sin modificaciones;
- no existan edades menores o iguales a cero en `df_clean`;
- las fechas tengan el tipo correcto;
- `tenure_days` no contenga valores negativos;
- los nulos numéricos permanezcan disponibles para el pipeline;
- las nuevas variables hayan sido creadas.

In [23]:
columnas_nuevas = [
    "coupon_code_present_flag",
    "date_inconsistency_flag",
    "tenure_days",
    "signup_year",
    "last_purchase_year",
    "location_inconsistency_flag",
    "total_spent_log"
]

print("Dimensiones de df original:", df.shape)
print("Dimensiones de df_clean:", df_clean.shape)

print("\nDuplicados exactos antes:", duplicados_antes)
print(
    "Duplicados exactos después:",
    int(df_clean.duplicated().sum())
)

print(
    "\nEdades <= 0 conservadas en df original:",
    int((df["age"] <= 0).sum())
)
print(
    "Edades <= 0 en df_clean:",
    int((df_clean["age"] <= 0).sum())
)

print("\nTipos de fecha:")
print(df_clean[["signup_date", "last_purchase_date"]].dtypes)

print(
    "\nInconsistencias temporales:",
    int(df_clean["date_inconsistency_flag"].eq(1).sum())
)
print(
    "Inconsistencias internas de ubicación:",
    int(df_clean["location_inconsistency_flag"].eq(1).sum())
)
print(
    "Antigüedades negativas:",
    int((df_clean["tenure_days"] < 0).sum())
)

print("\nColumnas nuevas:")
print(columnas_nuevas)

print("\nValores nulos restantes:")
nulos_restantes = (
    df_clean.isna().sum()
    .loc[lambda serie: serie > 0]
    .sort_values(ascending=False)
    .to_frame("Cantidad")
)
display(nulos_restantes)

Dimensiones de df original: (15000, 30)
Dimensiones de df_clean: (15000, 37)

Duplicados exactos antes: 0


Duplicados exactos después: 0

Edades <= 0 conservadas en df original: 4
Edades <= 0 en df_clean: 0

Tipos de fecha:
signup_date           datetime64[us]
last_purchase_date    datetime64[us]
dtype: object

Inconsistencias temporales: 3762
Inconsistencias internas de ubicación: 11818
Antigüedades negativas: 0

Columnas nuevas:
['coupon_code_present_flag', 'date_inconsistency_flag', 'tenure_days', 'signup_year', 'last_purchase_year', 'location_inconsistency_flag', 'total_spent_log']

Valores nulos restantes:


,Cantidad
tenure_days,3762
age,1204
total_spent,1050
total_spent_log,1050
satisfaction_score,702


### 18.7 Exportación del dataset intermedio

El resultado de la limpieza se guarda en
`data/interim/customers_clean.csv`.

El archivo original se mantiene sin modificaciones. Este dataset intermedio
servirá como base para incorporar posteriormente información externa y
continuar con el análisis del abandono de clientes.

In [24]:
ruta_raiz_proyecto = ruta_dataset.parents[2]
carpeta_interim = ruta_raiz_proyecto / "data" / "interim"
carpeta_interim.mkdir(parents=True, exist_ok=True)

ruta_dataset_intermedio = (
    carpeta_interim / "customers_clean.csv"
)

df_clean.to_csv(
    ruta_dataset_intermedio,
    index=False
)

dimensiones_exportadas = pd.read_csv(
    ruta_dataset_intermedio
).shape

print("Dataset intermedio exportado correctamente.")
print("Ruta:", ruta_dataset_intermedio)
print("Dimensiones exportadas:", dimensiones_exportadas)

Dataset intermedio exportado correctamente.
Ruta: ..\data\interim\customers_clean.csv
Dimensiones exportadas: (15000, 37)


## Conclusión de la limpieza e ingeniería inicial

La limpieza conserva los 15.000 clientes porque la auditoría no detectó
duplicados exactos.

Las cuatro edades menores o iguales a cero se transforman en valores faltantes,
sin modificar edades que requieren una política de negocio adicional.

Las 3.762 secuencias temporales incoherentes quedan identificadas mediante una
bandera y no generan antigüedades negativas.

La regla interna de país y ciudad marca 11.818 asociaciones inusuales. Esta
cantidad corresponde a la salida real de la auditoría vigente y debe
interpretarse como una inconsistencia interna, no como una validación geográfica
externa.

Los valores extremos de `total_spent` se conservan y se añade una versión
logarítmica. Las variables numéricas faltantes no se imputan en esta fase para
evitar fuga de información.

El resultado queda disponible en `data/interim/customers_clean.csv` para la
posterior integración de indicadores externos y las siguientes etapas del
análisis.

## 19. Integración de indicadores externos

Para complementar el análisis se incorporan dos indicadores de contexto
provenientes de la API V2 del Banco Mundial:

- porcentaje de población usuaria de Internet (`IT.NET.USER.ZS`);
- PIB per cápita en dólares estadounidenses corrientes (`NY.GDP.PCAP.CD`).

La integración se realiza por país y por el año de registro del cliente. Estos
indicadores describen el contexto nacional y no representan características
individuales ni demuestran causas del abandono.

### 19.1 Preparación de países, años y rutas

Se revisan los países y años presentes en el dataset limpio.

Los nombres de países se homologan a códigos ISO3 utilizados por el Banco
Mundial. Si aparece un país sin correspondencia, el proceso se detiene para
evitar uniones incompletas o asignaciones inventadas.

También se preparan las rutas para almacenar el caché de la API y el dataset
procesado.

In [25]:
from datetime import datetime, timezone

import requests

mapa_codigos_banco_mundial = {
    "Germany": "DEU",
    "India": "IND",
    "Bangladesh": "BGD",
    "USA": "USA",
    "UK": "GBR"
}

paises_dataset = sorted(
    df_clean["country"].dropna().unique().tolist()
)

paises_sin_codigo = sorted(
    set(paises_dataset)
    - set(mapa_codigos_banco_mundial)
)

if paises_sin_codigo:
    raise ValueError(
        "Existen países sin código del Banco Mundial: "
        f"{paises_sin_codigo}"
    )

anios_dataset = sorted(
    df_clean["signup_year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

if not anios_dataset:
    raise ValueError(
        "No existen años válidos en signup_year."
    )

ruta_raiz_proyecto = ruta_dataset.parents[2]

carpeta_external = (
    ruta_raiz_proyecto / "data" / "external"
)
carpeta_processed = (
    ruta_raiz_proyecto / "data" / "processed"
)

carpeta_external.mkdir(parents=True, exist_ok=True)
carpeta_processed.mkdir(parents=True, exist_ok=True)

ruta_cache_banco_mundial = (
    carpeta_external / "world_bank_indicators.csv"
)
ruta_dataset_procesado = (
    carpeta_processed / "customer_churn_processed.csv"
)

tabla_homologacion = pd.DataFrame({
    "country": paises_dataset,
    "country_code": [
        mapa_codigos_banco_mundial[pais]
        for pais in paises_dataset
    ]
})

display(tabla_homologacion)

print("Años de registro disponibles:", anios_dataset)
print("Ruta del caché:", ruta_cache_banco_mundial)
print("Ruta del dataset procesado:", ruta_dataset_procesado)

,country,country_code
0,Bangladesh,BGD
1,Germany,DEU
2,India,IND
3,UK,GBR
4,USA,USA


Años de registro disponibles: [2022, 2023, 2024]
Ruta del caché: ..\data\external\world_bank_indicators.csv
Ruta del dataset procesado: ..\data\processed\customer_churn_processed.csv


### 19.2 Consulta controlada de la API

La consulta utiliza únicamente los países y años presentes en el dataset, sin
realizar solicitudes por cada cliente.

Para reducir el impacto de fallas temporales del servicio se utiliza una sesión
HTTP con reintentos automáticos y espera progresiva.

La función incorpora:

- tiempo máximo separado para conexión y lectura;
- reintentos ante fallas temporales de red o del servidor;
- validación del código HTTP;
- validación básica de la estructura JSON;
- registro de la fecha de consulta;
- conservación de los valores faltantes publicados por la fuente.

Si la fuente no responde después de los reintentos, el flujo intenta utilizar
el caché local disponible.

In [26]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


sesion_banco_mundial = requests.Session()

sesion_banco_mundial.headers.update({
    "User-Agent": (
        "Proyecto-Universitario-Churn/1.0 "
        "(integracion-indicadores-banco-mundial)"
    )
})


configuracion_reintentos = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=2,
    status_forcelist=(
        429,
        500,
        502,
        503,
        504
    ),
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
    raise_on_status=False
)


adaptador_http = HTTPAdapter(
    max_retries=configuracion_reintentos
)


sesion_banco_mundial.mount(
    "https://",
    adaptador_http
)

sesion_banco_mundial.mount(
    "http://",
    adaptador_http
)


def consultar_indicador_banco_mundial(
    codigos_pais,
    codigo_indicador,
    anio_inicio,
    anio_fin,
    timeout_conexion=10,
    timeout_lectura=60
):
    if not codigos_pais:
        raise ValueError(
            "No se proporcionaron países para consultar."
        )

    codigos_consulta = ";".join(codigos_pais)

    url = (
        "https://api.worldbank.org/v2/country/"
        f"{codigos_consulta}/indicator/{codigo_indicador}"
    )

    parametros = {
        "format": "json",
        "date": f"{anio_inicio}:{anio_fin}",
        "per_page": 100,
        "source": 2
    }

    respuesta = sesion_banco_mundial.get(
        url,
        params=parametros,
        timeout=(
            timeout_conexion,
            timeout_lectura
        )
    )

    respuesta.raise_for_status()

    try:
        contenido = respuesta.json()
    except requests.exceptions.JSONDecodeError as error_json:
        raise ValueError(
            "La API respondió, pero el contenido no es JSON válido."
        ) from error_json

    estructura_valida = (
        isinstance(contenido, list)
        and len(contenido) >= 2
        and isinstance(contenido[0], dict)
        and isinstance(contenido[1], list)
    )

    if not estructura_valida:
        raise ValueError(
            "La API no devolvió la estructura JSON esperada."
        )

    metadata = contenido[0]

    fecha_consulta_utc = datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds")

    registros = []

    for observacion in contenido[1]:
        codigo_pais = observacion.get(
            "countryiso3code"
        )

        anio = observacion.get("date")

        if not codigo_pais or anio is None:
            raise ValueError(
                "La respuesta contiene una observación "
                "sin país o año."
            )

        try:
            anio = int(anio)
        except (TypeError, ValueError) as error_anio:
            raise ValueError(
                "La API devolvió un año inválido."
            ) from error_anio

        if codigo_pais not in codigos_pais:
            continue

        if not anio_inicio <= anio <= anio_fin:
            continue

        registros.append({
            "country_code": codigo_pais,
            "world_bank_country": (
                observacion.get("country", {})
                .get("value")
            ),
            "indicator_year": anio,
            "indicator_code": codigo_indicador,
            "indicator_name": (
                observacion.get("indicator", {})
                .get("value")
            ),
            "indicator_value": observacion.get(
                "value"
            ),
            "api_last_updated": metadata.get(
                "lastupdated"
            ),
            "query_utc": fecha_consulta_utc
        })

    if not registros:
        raise ValueError(
            "No se recibieron observaciones para el indicador "
            f"{codigo_indicador}."
        )

    resultado = pd.DataFrame(registros)

    resultado["indicator_value"] = pd.to_numeric(
        resultado["indicator_value"],
        errors="coerce"
    )

    if resultado.duplicated(
        [
            "country_code",
            "indicator_year"
        ]
    ).any():
        raise ValueError(
            "La API devolvió más de una observación "
            "para el mismo país y año."
        )

    resultado = (
        resultado
        .sort_values(
            [
                "country_code",
                "indicator_year"
            ]
        )
        .reset_index(drop=True)
    )

    return resultado

### Evidencia de conectividad

La conectividad fue comprobada directamente el 15 de julio de 2026 desde el
entorno virtual del proyecto. Los indicadores `IT.NET.USER.ZS` y
`NY.GDP.PCAP.CD` respondieron con estado HTTP 200 y entregaron 15 observaciones
válidas cada uno, sin combinaciones país-año duplicadas ni valores faltantes.
La API informó `2026-07-13` como fecha de última actualización.

Esta prueba confirma la disponibilidad de la fuente. El notebook mantiene el
caché local como mecanismo de contingencia y registra en cada ejecución si la
fuente utilizada fue la API o el caché.

### 19.3 Descarga y caché local

Se consultan el porcentaje de población usuaria de Internet y el PIB per cápita
para las combinaciones de país y año requeridas.

Cuando las respuestas son válidas, se construye una tabla con una fila por país
y año y se guarda en `data/external/world_bank_indicators.csv`.

Antes de guardar el archivo se comprueba que:

- estén presentes las columnas esperadas;
- cada combinación de país y año sea única;
- el caché cubra todas las combinaciones solicitadas;
- los años y los indicadores tengan tipos válidos.

Si la consulta falla, se utiliza el caché local existente. Cuando no existe un
caché válido, el proceso se detiene en lugar de inventar o completar
indicadores.

In [27]:
codigos_consulta = sorted(
    mapa_codigos_banco_mundial.values()
)

anio_inicio = min(anios_dataset)
anio_fin = max(anios_dataset)


columnas_cache_esperadas = [
    "country",
    "country_code",
    "indicator_year",
    "internet_users_pct",
    "gdp_per_capita_usd",
    "internet_api_last_updated",
    "gdp_api_last_updated",
    "internet_query_utc",
    "gdp_query_utc"
]


codigo_a_pais = {
    codigo: pais
    for pais, codigo
    in mapa_codigos_banco_mundial.items()
}


grilla_pais_anio = (
    pd.MultiIndex.from_product(
        [
            codigos_consulta,
            anios_dataset
        ],
        names=[
            "country_code",
            "indicator_year"
        ]
    )
    .to_frame(index=False)
)


grilla_pais_anio["country"] = (
    grilla_pais_anio["country_code"]
    .map(codigo_a_pais)
)


try:
    datos_internet = consultar_indicador_banco_mundial(
        codigos_pais=codigos_consulta,
        codigo_indicador="IT.NET.USER.ZS",
        anio_inicio=anio_inicio,
        anio_fin=anio_fin
    )

    datos_pib = consultar_indicador_banco_mundial(
        codigos_pais=codigos_consulta,
        codigo_indicador="NY.GDP.PCAP.CD",
        anio_inicio=anio_inicio,
        anio_fin=anio_fin
    )

    internet_seleccionado = (
        datos_internet[
            [
                "country_code",
                "indicator_year",
                "indicator_value",
                "api_last_updated",
                "query_utc"
            ]
        ]
        .rename(columns={
            "indicator_value": (
                "internet_users_pct"
            ),
            "api_last_updated": (
                "internet_api_last_updated"
            ),
            "query_utc": (
                "internet_query_utc"
            )
        })
    )

    pib_seleccionado = (
        datos_pib[
            [
                "country_code",
                "indicator_year",
                "indicator_value",
                "api_last_updated",
                "query_utc"
            ]
        ]
        .rename(columns={
            "indicator_value": (
                "gdp_per_capita_usd"
            ),
            "api_last_updated": (
                "gdp_api_last_updated"
            ),
            "query_utc": (
                "gdp_query_utc"
            )
        })
    )

    indicadores_banco_mundial = (
        grilla_pais_anio
        .merge(
            internet_seleccionado,
            on=[
                "country_code",
                "indicator_year"
            ],
            how="left",
            validate="one_to_one"
        )
        .merge(
            pib_seleccionado,
            on=[
                "country_code",
                "indicator_year"
            ],
            how="left",
            validate="one_to_one"
        )
    )

    origen_indicadores = (
        "API del Banco Mundial"
    )

except (
    requests.RequestException,
    ValueError,
    KeyError,
    TypeError
) as error_api:
    print(
        "No fue posible obtener los indicadores "
        "desde la API."
    )

    print(
        "Detalle:",
        type(error_api).__name__,
        "-",
        str(error_api)
    )

    if not ruta_cache_banco_mundial.exists():
        raise RuntimeError(
            "La consulta al Banco Mundial falló y todavía "
            "no existe un caché local. Reintenta con conexión "
            "a Internet antes de continuar."
        ) from error_api

    try:
        indicadores_banco_mundial = pd.read_csv(
            ruta_cache_banco_mundial
        )
    except (
        OSError,
        pd.errors.EmptyDataError,
        pd.errors.ParserError
    ) as error_cache:
        raise RuntimeError(
            "Existe un archivo de caché, pero no pudo "
            "ser leído correctamente."
        ) from error_cache

    origen_indicadores = "caché local"


columnas_faltantes_cache = sorted(
    set(columnas_cache_esperadas)
    - set(indicadores_banco_mundial.columns)
)


if columnas_faltantes_cache:
    raise ValueError(
        "El archivo externo no contiene las columnas "
        f"esperadas: {columnas_faltantes_cache}"
    )


indicadores_banco_mundial[
    "indicator_year"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "indicator_year"
    ],
    errors="raise"
).astype("Int64")


indicadores_banco_mundial[
    "internet_users_pct"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "internet_users_pct"
    ],
    errors="coerce"
)


indicadores_banco_mundial[
    "gdp_per_capita_usd"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "gdp_per_capita_usd"
    ],
    errors="coerce"
)


indicadores_banco_mundial = (
    indicadores_banco_mundial.loc[
        indicadores_banco_mundial[
            "country_code"
        ].isin(codigos_consulta)
        & indicadores_banco_mundial[
            "indicator_year"
        ].isin(anios_dataset)
    ]
    .copy()
)


if indicadores_banco_mundial.duplicated(
    [
        "country_code",
        "indicator_year"
    ]
).any():
    raise ValueError(
        "El archivo externo contiene países y años duplicados."
    )


combinaciones_esperadas = set(
    grilla_pais_anio[
        [
            "country_code",
            "indicator_year"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)


combinaciones_obtenidas = set(
    indicadores_banco_mundial[
        [
            "country_code",
            "indicator_year"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)


combinaciones_faltantes = sorted(
    combinaciones_esperadas
    - combinaciones_obtenidas
)


if combinaciones_faltantes:
    raise ValueError(
        "La información externa no cubre todas las "
        "combinaciones país-año requeridas: "
        f"{combinaciones_faltantes}"
    )


indicadores_banco_mundial = (
    indicadores_banco_mundial[
        columnas_cache_esperadas
    ]
    .sort_values(
        [
            "country",
            "indicator_year"
        ]
    )
    .reset_index(drop=True)
)


if origen_indicadores == "API del Banco Mundial":
    indicadores_banco_mundial.to_csv(
        ruta_cache_banco_mundial,
        index=False
    )


print("Fuente utilizada:", origen_indicadores)

print(
    "Última actualización de Internet:",
    sorted(
        indicadores_banco_mundial[
            "internet_api_last_updated"
        ].dropna().astype(str).unique().tolist()
    )
)

print(
    "Última actualización de PIB:",
    sorted(
        indicadores_banco_mundial[
            "gdp_api_last_updated"
        ].dropna().astype(str).unique().tolist()
    )
)

print(
    "Fecha de consulta de Internet:",
    sorted(
        indicadores_banco_mundial[
            "internet_query_utc"
        ].dropna().astype(str).unique().tolist()
    )
)

print(
    "Fecha de consulta de PIB:",
    sorted(
        indicadores_banco_mundial[
            "gdp_query_utc"
        ].dropna().astype(str).unique().tolist()
    )
)

print(
    "Dimensiones de los indicadores:",
    indicadores_banco_mundial.shape
)

print(
    "Ruta del caché:",
    ruta_cache_banco_mundial
)

display(indicadores_banco_mundial)

Fuente utilizada: API del Banco Mundial
Última actualización de Internet: ['2026-07-13']
Última actualización de PIB: ['2026-07-13']
Fecha de consulta de Internet: ['2026-07-15T22:39:51+00:00']
Fecha de consulta de PIB: ['2026-07-15T22:39:51+00:00']
Dimensiones de los indicadores: (15, 9)
Ruta del caché: ..\data\external\world_bank_indicators.csv


,country,country_code,indicator_year,internet_users_pct,gdp_per_capita_usd,internet_api_last_updated,gdp_api_last_updated,internet_query_utc,gdp_query_utc
0,Bangladesh,BGD,2022,41.62,"2,716.49",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
1,Bangladesh,BGD,2023,44.50,"2,551.02",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
2,Bangladesh,BGD,2024,53.42,"2,593.42",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
3,Germany,DEU,2022,91.63,"50,506.52",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
4,Germany,DEU,2023,92.48,"54,776.77",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
5,Germany,DEU,2024,93.50,"56,103.73",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
6,India,IND,2022,55.90,"2,279.98",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
7,India,IND,2023,60.25,"2,434.45",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
8,India,IND,2024,64.94,"2,591.99",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00
9,UK,GBR,2022,95.47,"47,034.78",2026-07-13,2026-07-13,2026-07-15T22:39:51+00:00,2026-07-15T22:39:51+00:00


### 19.4 Revisión de disponibilidad

La tabla externa debe contener una combinación única por país y año.

Los valores faltantes se conservan porque algunos indicadores pueden publicarse
con retraso. No se reemplazan con el año anterior ni con promedios, ya que eso
alteraría la correspondencia temporal.

In [28]:
resumen_disponibilidad = (
    indicadores_banco_mundial
    .groupby("country", as_index=False)
    .agg(
        anios_consultados=(
            "indicator_year",
            "nunique"
        ),
        internet_disponible=(
            "internet_users_pct",
            lambda serie: int(serie.notna().sum())
        ),
        pib_disponible=(
            "gdp_per_capita_usd",
            lambda serie: int(serie.notna().sum())
        )
    )
)

print(
    "Combinaciones país-año esperadas:",
    len(codigos_consulta) * len(anios_dataset)
)

print(
    "Combinaciones país-año obtenidas:",
    len(indicadores_banco_mundial)
)

print(
    "Valores faltantes en uso de Internet:",
    int(
        indicadores_banco_mundial[
            "internet_users_pct"
        ].isna().sum()
    )
)

print(
    "Valores faltantes en PIB per cápita:",
    int(
        indicadores_banco_mundial[
            "gdp_per_capita_usd"
        ].isna().sum()
    )
)

display(resumen_disponibilidad)

Combinaciones país-año esperadas: 15
Combinaciones país-año obtenidas: 15
Valores faltantes en uso de Internet: 0
Valores faltantes en PIB per cápita: 0


,country,anios_consultados,internet_disponible,pib_disponible
0,Bangladesh,3,3,3
1,Germany,3,3,3
2,India,3,3,3
3,UK,3,3,3
4,USA,3,3,3


### 19.5 Unión con los clientes

Los indicadores se unen con `df_clean` mediante:

- `country_code`, derivado de `country`;
- `signup_year`, derivado de `signup_date`.

La unión es de tipo izquierda para conservar a todos los clientes. La validación
`many_to_one` garantiza que cada combinación de país y año corresponda como
máximo a una fila de indicadores.

In [29]:
df_processed = df_clean.copy()

df_processed["country_code"] = (
    df_processed["country"]
    .map(mapa_codigos_banco_mundial)
)

if df_processed["country_code"].isna().any():
    paises_sin_union = sorted(
        df_processed.loc[
            df_processed["country_code"].isna(),
            "country"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Existen países sin homologación: "
        f"{paises_sin_union}"
    )

filas_antes_union = len(df_processed)

df_processed = df_processed.merge(
    indicadores_banco_mundial[
        [
            "country_code",
            "indicator_year",
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ],
    left_on=["country_code", "signup_year"],
    right_on=["country_code", "indicator_year"],
    how="left",
    validate="many_to_one"
)

filas_despues_union = len(df_processed)

if filas_antes_union != filas_despues_union:
    raise ValueError(
        "La unión modificó la cantidad de clientes."
    )

combinaciones_sin_coincidencia = (
    df_processed.loc[
        df_processed["indicator_year"].isna(),
        ["country", "signup_year"]
    ]
    .drop_duplicates()
    .sort_values(["country", "signup_year"])
)

print("Filas antes de la unión:", filas_antes_union)
print("Filas después de la unión:", filas_despues_union)

print(
    "Clientes sin coincidencia país-año:",
    int(df_processed["indicator_year"].isna().sum())
)

print(
    "Clientes sin dato de uso de Internet:",
    int(df_processed["internet_users_pct"].isna().sum())
)

print(
    "Clientes sin dato de PIB per cápita:",
    int(df_processed["gdp_per_capita_usd"].isna().sum())
)

if not combinaciones_sin_coincidencia.empty:
    display(combinaciones_sin_coincidencia)

df_processed = df_processed.drop(
    columns="indicator_year"
)

Filas antes de la unión: 15000
Filas después de la unión: 15000
Clientes sin coincidencia país-año: 0
Clientes sin dato de uso de Internet: 0
Clientes sin dato de PIB per cápita: 0


### 19.6 Validación de la integración

Se verifica que:

- la cantidad de clientes no cambie;
- los identificadores permanezcan únicos;
- no existan países sin código;
- la unión no genere filas duplicadas;
- los indicadores externos conserven sus valores faltantes reales.

Los indicadores se mantendrán inicialmente como variables de contexto. Su
incorporación al modelo deberá justificarse y compararse posteriormente.

In [30]:
print("Dimensiones de df_clean:", df_clean.shape)
print("Dimensiones de df_processed:", df_processed.shape)

print(
    "Clientes únicos en df_processed:",
    df_processed["customer_id"].nunique()
)

print(
    "Identificadores duplicados:",
    int(
        df_processed[
            "customer_id"
        ].duplicated().sum()
    )
)

print(
    "Países sin código:",
    int(df_processed["country_code"].isna().sum())
)

columnas_externas = [
    "country_code",
    "internet_users_pct",
    "gdp_per_capita_usd"
]

print("\nColumnas externas incorporadas:")
print(columnas_externas)

print("\nResumen de indicadores externos:")
display(
    df_processed[
        [
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ].describe().T
)

Dimensiones de df_clean: (15000, 37)
Dimensiones de df_processed: (15000, 40)
Clientes únicos en df_processed: 15000
Identificadores duplicados: 0
Países sin código: 0

Columnas externas incorporadas:
['country_code', 'internet_users_pct', 'gdp_per_capita_usd']

Resumen de indicadores externos:


,count,mean,std,min,25%,50%,75%,max
internet_users_pct,"15,000.00",77.48,20.67,41.62,55.90,92.48,93.53,95.47
gdp_per_capita_usd,"15,000.00","38,006.06","31,078.98","2,279.98","2,591.99","49,919.69","56,103.73","86,169.66"


### 19.7 Exportación del dataset procesado

El dataset enriquecido se guarda en
`data/processed/customer_churn_processed.csv`.

El archivo conserva a todos los clientes y añade los indicadores externos sin
modificar los archivos ubicados en `data/raw` ni el dataset intermedio.

In [31]:
df_processed.to_csv(
    ruta_dataset_procesado,
    index=False
)

dimensiones_procesadas = pd.read_csv(
    ruta_dataset_procesado
).shape

print("Dataset procesado exportado correctamente.")
print("Ruta:", ruta_dataset_procesado)
print("Dimensiones exportadas:", dimensiones_procesadas)

print(
    "Caché externo disponible:",
    ruta_cache_banco_mundial.exists()
)

Dataset procesado exportado correctamente.
Ruta: ..\data\processed\customer_churn_processed.csv
Dimensiones exportadas: (15000, 40)
Caché externo disponible: True


## Conclusión de la integración de datos externos

Los clientes fueron enriquecidos mediante indicadores del Banco Mundial
consultados por país y año de registro.

La consulta se realiza por valores únicos y utiliza un caché local como respaldo
ante fallas de conexión. La unión conserva la cantidad original de clientes y
mantiene como faltantes los indicadores que la fuente no publica para una
combinación específica.

El porcentaje de población usuaria de Internet y el PIB per cápita representan
contexto nacional. Por ello, sus asociaciones con `churn` deberán interpretarse
con cautela y no como relaciones causales individuales.

El dataset procesado queda disponible en
`data/processed/customer_churn_processed.csv` para continuar con el análisis
exploratorio y la preparación del modelado.

## 20. Análisis exploratorio interactivo

Esta etapa examina relaciones descriptivas entre el abandono y variables relevantes para marketing, retención, experiencia del cliente y soporte.

El análisis utiliza el dataset procesado exportado después de la integración externa. No se imputan valores faltantes, no se eliminan valores extremos y no se aplican transformaciones destinadas al modelado.

In [32]:
pip install plotly pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 20.1 Validación del dataset procesado

Antes de construir visualizaciones se comprueba la estructura persistida, la unicidad de clientes, la distribución de la variable objetivo, los valores faltantes y la disponibilidad de los indicadores externos.

In [34]:
import plotly.express as px

df_eda = pd.read_csv(ruta_dataset_procesado)

columnas_requeridas_eda = [
    "customer_id",
    "churn",
    "acquisition_channel",
    "satisfaction_score",
    "support_tickets",
    "total_spent_log",
    "country",
    "signup_year",
    "internet_users_pct",
    "gdp_per_capita_usd"
]

columnas_faltantes_eda = sorted(
    set(columnas_requeridas_eda)
    - set(df_eda.columns)
)

if columnas_faltantes_eda:
    raise ValueError(
        "Faltan columnas necesarias para el análisis: "
        f"{columnas_faltantes_eda}"
    )

if df_eda["customer_id"].duplicated().any():
    raise ValueError(
        "El dataset procesado contiene identificadores duplicados."
    )

print(df_eda.shape)
print(df_eda.columns.tolist())

print("\nDistribución absoluta de churn:")
print(
    df_eda["churn"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDistribución relativa de churn:")
print(
    df_eda["churn"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .sort_index()
)

print("\nPrincipales valores faltantes:")
print(
    df_eda.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nClientes por país:")
print(
    df_eda["country"]
    .value_counts(dropna=False)
)

print("\nClientes por año de registro:")
print(
    df_eda["signup_year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nValores faltantes en indicadores externos:")
print(
    df_eda[
        [
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ].isna().sum()
)

print(
    "\nFilas duplicadas:",
    int(df_eda.duplicated().sum())
)

print(
    "Clientes únicos:",
    int(
        df_eda["customer_id"]
        .nunique(dropna=False)
    )
)

(15000, 40)
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type', 'is_premium_user', 'total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'payment_method', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn', 'coupon_code_present_flag', 'date_inconsistency_flag', 'tenure_days', 'signup_year', 'last_purchase_year', 'location_inconsistency_flag', 'total_spent_log', 'country_code', 'internet_users_pct', 'gdp_per_capita_usd']

Distribución absoluta de churn:
churn
0    12702
1     2298
Name: count, dtype: int64

Distribución relativa de churn:
churn
0   0.85
1   0.15
Name: proportion, dtype: float64

Principales valores faltantes:
tenure_days            3762
age  

**Interpretación.** El archivo procesado conserva 15.000 clientes y 40 columnas. Los 15.000 identificadores son únicos y no existen filas duplicadas. La clase de abandono representa el 15,32 % de los clientes, por lo que existe un desbalance relevante.

Los principales valores faltantes permanecen en `tenure_days`, `age`, `total_spent`, `total_spent_log` y `satisfaction_score`. Los indicadores externos no presentan valores faltantes para los cinco países y los años 2022, 2023 y 2024.

**Utilidad.** Estas comprobaciones permiten interpretar las tasas sobre una base estable y evitan confundir diferencias de tamaño entre grupos con diferencias de riesgo.

**Limitación.** La ausencia de valores se conserva sin imputación. Por ello, cada gráfico de una variable incompleta debe informar cuántas observaciones válidas utiliza.

### 20.2 ¿Qué proporción de clientes abandonó?

La distribución de la variable objetivo permite dimensionar el problema antes de comparar segmentos.


In [35]:
tabla_churn = (
    df_eda["churn"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("churn")
    .reset_index(name="total_customers")
)

tabla_churn["churn_label"] = (
    tabla_churn["churn"]
    .map({
        0: "No abandonó",
        1: "Abandonó"
    })
)

tabla_churn["percentage"] = (
    tabla_churn["total_customers"]
    / len(df_eda)
    * 100
)

display(
    tabla_churn[
        [
            "churn_label",
            "total_customers",
            "percentage"
        ]
    ]
)

fig_churn = px.bar(
    tabla_churn,
    x="churn_label",
    y="percentage",
    text_auto=".2f",
    hover_data={
        "total_customers": True,
        "percentage": ":.2f"
    },
    labels={
        "churn_label": "Estado del cliente",
        "percentage": "Clientes (%)",
        "total_customers": "Cantidad de clientes"
    },
    title="Distribución de clientes según abandono"
)

fig_churn.update_layout(
    yaxis_range=[
        0,
        tabla_churn["percentage"].max() * 1.12
    ]
)

fig_churn.show()

,churn_label,total_customers,percentage
0,No abandonó,12702,84.68
1,Abandonó,2298,15.32


**Interpretación.** De los 15.000 clientes, 12.702 no abandonaron y 2.298 abandonaron. La tasa observada de churn es 15,32 %, mientras que la clase mayoritaria concentra 84,68 %.

**Utilidad.** Marketing y retención deben evaluar resultados utilizando tasas y no solo cantidades absolutas. La evaluación predictiva posterior deberá considerar este desbalance al interpretar la accuracy.

**Limitación.** Esta distribución describe la frecuencia del abandono, pero no identifica qué variables están asociadas con el fenómeno.

### 20.3 ¿Qué asociaciones lineales aparecen entre las variables numéricas?

Se seleccionan variables representativas del comportamiento, gasto, experiencia y soporte.

Se excluye `customer_id`, se utiliza `total_spent_log` en lugar de incluir simultáneamente ambas escalas de gasto y los indicadores nacionales se reservan para un análisis agregado por país y año.

In [36]:
variables_correlacion = [
    "age",
    "total_visits",
    "avg_session_time",
    "pages_per_session",
    "email_open_rate",
    "email_click_rate",
    "total_spent_log",
    "avg_order_value",
    "support_tickets",
    "delivery_delay_days",
    "satisfaction_score",
    "nps_score",
    "last_3_month_purchase_freq",
    "churn"
]

matriz_correlacion = (
    df_eda[variables_correlacion]
    .corr(method="pearson")
)

fig_correlacion = px.imshow(
    matriz_correlacion,
    text_auto=".2f",
    aspect="auto",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title=(
        "Correlaciones lineales entre variables "
        "numéricas seleccionadas"
    )
)

fig_correlacion.update_layout(
    height=760
)

fig_correlacion.show()

correlacion_con_churn = (
    matriz_correlacion["churn"]
    .drop("churn")
    .sort_values(
        key=lambda serie: serie.abs(),
        ascending=False
    )
    .rename("correlation_with_churn")
    .to_frame()
)

display(correlacion_con_churn)

,correlation_with_churn
total_spent_log,-0.32
satisfaction_score,-0.30
support_tickets,0.13
total_visits,0.01
avg_session_time,0.01
email_open_rate,-0.01
pages_per_session,0.01
nps_score,0.01
age,0.00
email_click_rate,-0.00


**Interpretación.** Las asociaciones lineales de mayor magnitud con `churn` corresponden a `total_spent_log`, `satisfaction_score` y `support_tickets`. En términos descriptivos, el abandono aparece asociado con menor gasto, menor satisfacción y una mayor cantidad de solicitudes de soporte.

**Utilidad.** Estas variables merecen revisión específica durante el análisis y posteriormente pueden evaluarse dentro del pipeline predictivo junto con el resto de características válidas.

**Limitación.** La correlación de Pearson representa principalmente asociación lineal y no implica causalidad. Como `churn` es binaria, estos coeficientes deben interpretarse como una referencia exploratoria. Una correlación baja tampoco demuestra que una variable sea inútil para un modelo no lineal o en interacción con otras variables.

### 20.4 ¿Cómo varía la tasa de churn por canal de adquisición?

Para comparar correctamente los canales se calcula la cantidad total de clientes, la cantidad de clientes churn y la tasa de churn de cada grupo.

In [37]:
tabla_canal = (
    df_eda
    .groupby(
        "acquisition_channel",
        as_index=False
    )["churn"]
    .agg(
        total_customers="size",
        churn_customers="sum",
        churn_rate="mean"
    )
)

tabla_canal["churn_rate_pct"] = (
    tabla_canal["churn_rate"] * 100
)

tabla_canal = tabla_canal.sort_values(
    "churn_rate_pct",
    ascending=False
)

display(
    tabla_canal[
        [
            "acquisition_channel",
            "total_customers",
            "churn_customers",
            "churn_rate_pct"
        ]
    ]
)

fig_canal = px.bar(
    tabla_canal,
    x="acquisition_channel",
    y="churn_rate_pct",
    text_auto=".2f",
    custom_data=[
        "total_customers",
        "churn_customers"
    ],
    labels={
        "acquisition_channel": "Canal de adquisición",
        "churn_rate_pct": "Tasa de churn (%)"
    },
    title="Tasa de churn por canal de adquisición"
)

fig_canal.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Tasa de churn: %{y:.2f}%<br>"
        "Clientes: %{customdata[0]}<br>"
        "Clientes churn: %{customdata[1]}"
        "<extra></extra>"
    )
)

fig_canal.show()

,acquisition_channel,total_customers,churn_customers,churn_rate_pct
3,Organic,3055,485,15.88
4,Referral,2941,464,15.78
2,Google Ads,3047,462,15.16
0,Email,2933,439,14.97
1,Facebook Ads,3024,448,14.81


**Interpretación.** Las tasas por canal son cercanas. `Organic` presenta 15,88 % y `Referral` 15,78 %, mientras que `Facebook Ads` registra 14,81 %. La diferencia entre el máximo y el mínimo es de aproximadamente 1,06 puntos porcentuales, por lo que el canal de adquisición no separa fuertemente a los clientes en este análisis descriptivo.

**Utilidad.** No conviene reasignar inversión de marketing utilizando solo esta diferencia. El canal puede combinarse posteriormente con satisfacción, gasto, soporte y otras variables para identificar perfiles más específicos.

**Limitación.** Las tasas no controlan por composición demográfica, antigüedad, gasto u otras características. Las diferencias observadas no demuestran que un canal cause mayor o menor abandono.

### 20.5 ¿Qué relación descriptiva existe entre satisfacción y churn?

Se comparan las distribuciones completas y se informa la cantidad de valores válidos y faltantes. No se imputan puntuaciones para construir el gráfico.

In [38]:
tabla_satisfaccion = (
    df_eda
    .groupby(
        "churn",
        as_index=False
    )
    .agg(
        total_customers=(
            "customer_id",
            "size"
        ),
        valid_satisfaction=(
            "satisfaction_score",
            "count"
        ),
        mean_satisfaction=(
            "satisfaction_score",
            "mean"
        ),
        median_satisfaction=(
            "satisfaction_score",
            "median"
        )
    )
)

tabla_satisfaccion[
    "missing_satisfaction"
] = (
    tabla_satisfaccion["total_customers"]
    - tabla_satisfaccion["valid_satisfaction"]
)

tabla_satisfaccion["missing_pct"] = (
    tabla_satisfaccion["missing_satisfaction"]
    / tabla_satisfaccion["total_customers"]
    * 100
)

tabla_satisfaccion["churn_label"] = (
    tabla_satisfaccion["churn"]
    .map({
        0: "No abandonó",
        1: "Abandonó"
    })
)

display(
    tabla_satisfaccion[
        [
            "churn_label",
            "total_customers",
            "valid_satisfaction",
            "missing_satisfaction",
            "missing_pct",
            "mean_satisfaction",
            "median_satisfaction"
        ]
    ]
)

satisfaccion_valida = (
    df_eda
    .dropna(
        subset=["satisfaction_score"]
    )
    .copy()
)

satisfaccion_valida["churn_label"] = (
    satisfaccion_valida["churn"]
    .map({
        0: "No abandonó",
        1: "Abandonó"
    })
)

fig_satisfaccion = px.box(
    satisfaccion_valida,
    x="churn_label",
    y="satisfaction_score",
    points=False,
    category_orders={
        "churn_label": [
            "No abandonó",
            "Abandonó"
        ]
    },
    labels={
        "churn_label": "Estado del cliente",
        "satisfaction_score": (
            "Puntuación de satisfacción"
        )
    },
    title=(
        "Distribución de satisfacción "
        "según abandono"
    )
)

fig_satisfaccion.show()

,churn_label,total_customers,valid_satisfaction,missing_satisfaction,missing_pct,mean_satisfaction,median_satisfaction
0,No abandonó,12702,12073,629,4.95,3.73,4.00
1,Abandonó,2298,2225,73,3.18,2.82,3.00


**Interpretación.** Los clientes que abandonaron presentan una satisfacción promedio de 2,82 y mediana de 3, frente a un promedio de 3,73 y mediana de 4 entre quienes no abandonaron. El gráfico utiliza 14.298 observaciones válidas y mantiene 702 valores faltantes sin imputar.

**Utilidad.** La satisfacción puede utilizarse como señal para activar encuestas de seguimiento, recuperación de servicio o contacto preventivo, especialmente cuando coincide con otras señales de riesgo.

**Limitación.** La asociación no establece dirección temporal. Una baja satisfacción podría preceder al abandono, coexistir con él o haber sido registrada después de experiencias que también influyen en churn.

### 20.6 ¿Cómo cambia la tasa de churn según las interacciones de soporte?

Las categorías con cinco o más tickets contienen pocos registros por valor individual. Para evitar tasas inestables se agrupan exclusivamente para esta visualización bajo la etiqueta `5+`, sin modificar la variable original.

In [39]:
soporte_eda = (
    df_eda[
        [
            "support_tickets",
            "churn"
        ]
    ]
    .copy()
)

soporte_eda["support_ticket_group"] = np.where(
    soporte_eda["support_tickets"] >= 5,
    "5+",
    soporte_eda["support_tickets"].astype(str)
)

orden_soporte = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5+"
]

soporte_eda["support_ticket_group"] = pd.Categorical(
    soporte_eda["support_ticket_group"],
    categories=orden_soporte,
    ordered=True
)

tabla_soporte = (
    soporte_eda
    .groupby(
        "support_ticket_group",
        observed=False
    )["churn"]
    .agg(
        total_customers="size",
        churn_customers="sum",
        churn_rate="mean"
    )
    .reset_index()
)

tabla_soporte["churn_rate_pct"] = (
    tabla_soporte["churn_rate"] * 100
)

display(
    tabla_soporte[
        [
            "support_ticket_group",
            "total_customers",
            "churn_customers",
            "churn_rate_pct"
        ]
    ]
)

fig_soporte = px.bar(
    tabla_soporte,
    x="support_ticket_group",
    y="churn_rate_pct",
    text_auto=".2f",
    custom_data=[
        "total_customers",
        "churn_customers"
    ],
    labels={
        "support_ticket_group": (
            "Cantidad de tickets"
        ),
        "churn_rate_pct": "Tasa de churn (%)"
    },
    title=(
        "Tasa de churn según cantidad "
        "de tickets de soporte"
    )
)

fig_soporte.update_traces(
    hovertemplate=(
        "<b>Tickets: %{x}</b><br>"
        "Tasa de churn: %{y:.2f}%<br>"
        "Clientes: %{customdata[0]}<br>"
        "Clientes churn: %{customdata[1]}"
        "<extra></extra>"
    )
)

fig_soporte.show()

,support_ticket_group,total_customers,churn_customers,churn_rate_pct
0,0,2071,264,12.75
1,1,4047,575,14.21
2,2,4069,537,13.20
3,3,2653,350,13.19
4,4,1351,164,12.14
5,5+,809,408,50.43


**Interpretación.** Los grupos entre cero y cuatro tickets presentan tasas entre 12,14 % y 14,21 %. En cambio, los 809 clientes con cinco o más tickets alcanzan una tasa de churn de 50,43 %, una diferencia descriptiva considerable.

**Utilidad.** El volumen elevado de solicitudes de soporte puede emplearse como criterio operativo para priorizar revisión de casos, recuperación de servicio y contacto de retención.

**Limitación.** La agrupación `5+` mejora la estabilidad visual, pero oculta diferencias internas entre valores altos. Además, la cantidad de tickets puede ser una consecuencia de problemas previos y no necesariamente una causa directa del abandono.

### 20.7 ¿Qué contexto aportan los indicadores nacionales?

Los indicadores del Banco Mundial se agregan por país y año antes de graficarse. De esta forma, cada combinación macroeconómica aparece una sola vez y no se trata a cada cliente como una observación nacional independiente.

In [40]:
tabla_contexto = (
    df_eda
    .groupby(
        [
            "country",
            "signup_year",
            "internet_users_pct",
            "gdp_per_capita_usd"
        ],
        as_index=False
    )
    .agg(
        total_customers=(
            "customer_id",
            "size"
        ),
        churn_customers=(
            "churn",
            "sum"
        ),
        churn_rate=(
            "churn",
            "mean"
        )
    )
)

tabla_contexto["churn_rate_pct"] = (
    tabla_contexto["churn_rate"] * 100
)

tabla_contexto["signup_year_label"] = (
    tabla_contexto["signup_year"]
    .astype(str)
)

if len(tabla_contexto) != 15:
    raise ValueError(
        "La tabla contextual no contiene las "
        "15 combinaciones país-año esperadas."
    )

display(
    tabla_contexto[
        [
            "country",
            "signup_year",
            "total_customers",
            "churn_customers",
            "churn_rate_pct",
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ]
    .sort_values(
        [
            "country",
            "signup_year"
        ]
    )
)

fig_contexto = px.scatter(
    tabla_contexto,
    x="gdp_per_capita_usd",
    y="churn_rate_pct",
    color="internet_users_pct",
    symbol="signup_year_label",
    size="total_customers",
    size_max=28,
    hover_name="country",
    hover_data={
        "signup_year": True,
        "gdp_per_capita_usd": ":,.2f",
        "internet_users_pct": ":.2f",
        "churn_rate_pct": ":.2f",
        "total_customers": True,
        "signup_year_label": False
    },
    labels={
        "gdp_per_capita_usd": (
            "PIB per cápita (USD corrientes)"
        ),
        "churn_rate_pct": "Tasa de churn (%)",
        "internet_users_pct": (
            "Usuarios de Internet (%)"
        ),
        "signup_year_label": "Año de registro"
    },
    title=(
        "Churn agregado por país y año "
        "frente al contexto nacional"
    )
)

fig_contexto.show()

,country,signup_year,total_customers,churn_customers,churn_rate_pct,internet_users_pct,gdp_per_capita_usd
0,Bangladesh,2022,1058,159,15.03,41.62,"2,716.49"
1,Bangladesh,2023,1100,171,15.55,44.50,"2,551.02"
2,Bangladesh,2024,826,134,16.22,53.42,"2,593.42"
3,Germany,2022,1159,169,14.58,91.63,"50,506.52"
4,Germany,2023,1088,131,12.04,92.48,"54,776.77"
5,Germany,2024,825,128,15.52,93.50,"56,103.73"
6,India,2022,1084,168,15.50,55.90,"2,279.98"
7,India,2023,1090,185,16.97,60.25,"2,434.45"
8,India,2024,840,127,15.12,64.94,"2,591.99"
9,UK,2022,1104,155,14.04,95.47,"47,034.78"


**Interpretación.** Las 15 combinaciones país-año presentan tasas de churn entre 12,04 % y 16,97 %. No se observa un patrón descriptivo uniforme en el que un mayor PIB per cápita o un mayor porcentaje de usuarios de Internet se relacione siempre con una tasa mayor o menor de abandono.

**Utilidad.** Los indicadores pueden aportar contexto al comparar mercados, pero no deben utilizarse como explicación individual ni como criterio aislado de intervención sobre clientes.

**Limitación.** Solo existen 15 observaciones agregadas y los valores nacionales se repiten entre clientes del mismo país y año. Cualquier relación observada está expuesta a falacia ecológica y no permite inferir causalidad.

### 20.8 Principales hallazgos

- La tasa general de churn es 15,32 %, por lo que la clase positiva es minoritaria.
- `total_spent_log` y `satisfaction_score` presentan las asociaciones lineales negativas de mayor magnitud con churn.
- `support_tickets` presenta una asociación positiva y una diferencia operativamente relevante en el grupo con cinco o más tickets.
- Los canales de adquisición muestran tasas próximas, entre 14,81 % y 15,88 %, por lo que no constituyen por sí solos una segmentación fuerte.
- Los clientes churn presentan menor satisfacción en las observaciones disponibles.
- Los indicadores nacionales aportan contexto agregado, pero no exhiben una relación uniforme ni permiten explicar el abandono individual.

### 20.9 Limitaciones del análisis exploratorio

Este análisis es descriptivo y observacional. No permite establecer causalidad ni dirección temporal entre las variables y el abandono.

Los valores faltantes se conservaron sin imputación, por lo que algunas comparaciones utilizan subconjuntos de observaciones válidas. La correlación de Pearson se limita principalmente a asociaciones lineales y no será el único criterio para seleccionar variables.

La agrupación de tickets en `5+` se utiliza solo para estabilizar la visualización. Los indicadores del Banco Mundial corresponden a contexto nacional y no a características individuales.

La separación entre entrenamiento y prueba, la imputación dentro del pipeline y la evaluación predictiva se realizarán posteriormente.

## 21. Preparación y modelado inicial

Esta etapa construye un primer flujo reproducible de clasificación binaria. La
separación entre entrenamiento y prueba se realiza antes de ajustar cualquier
imputación, escalamiento o codificación.

La regresión logística se utiliza como referencia inicial y no como modelo final.
Su rendimiento se compara con un clasificador que predice siempre la clase
mayoritaria.

### 21.1 Revisión de fuga de información

Antes de definir las variables predictoras se revisa qué información debe quedar
fuera del primer modelo.

`churn` corresponde a la variable objetivo y `customer_id` es un identificador
sin significado predictivo generalizable. Las fechas originales se reemplazan
por variables derivadas, `coupon_code` por una bandera de presencia y
`total_spent` por su transformación logarítmica.

`lifetime_value` se excluye de forma preventiva. La documentación disponible no
define con precisión cuándo se calcula y podría incorporar información acumulada
posterior al momento en que se desea predecir el abandono. Esta decisión reduce
el riesgo de fuga temporal, pero no demuestra que la variable contenga fuga.

Las banderas de calidad se conservan como variables exploratorias. Representan
patrones internos del dataset y no deben interpretarse como errores geográficos
o temporales comprobados.

In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df_model = pd.read_csv(ruta_dataset_procesado)

if df_model.empty:
    raise ValueError(
        "El dataset procesado está vacío y no permite preparar el modelo."
    )

columnas_basicas_modelo = [
    "customer_id",
    "churn"
]

columnas_basicas_faltantes = sorted(
    set(columnas_basicas_modelo)
    - set(df_model.columns)
)

if columnas_basicas_faltantes:
    raise ValueError(
        "Faltan columnas básicas para el modelado: "
        f"{columnas_basicas_faltantes}"
    )

if df_model["churn"].isna().any():
    raise ValueError(
        "La variable churn contiene valores faltantes."
    )

clases_churn = set(
    df_model["churn"]
    .dropna()
    .unique()
    .tolist()
)

if clases_churn != {0, 1}:
    raise ValueError(
        "La variable churn debe contener únicamente las clases 0 y 1. "
        f"Clases encontradas: {sorted(clases_churn)}"
    )

if df_model["customer_id"].isna().any():
    raise ValueError(
        "customer_id contiene valores faltantes."
    )

if df_model["customer_id"].duplicated().any():
    raise ValueError(
        "customer_id contiene identificadores duplicados."
    )

tabla_exclusiones_modelo = pd.DataFrame({
    "variable_excluida": [
        "churn",
        "customer_id",
        "city",
        "signup_date",
        "last_purchase_date",
        "coupon_code",
        "country_code",
        "total_spent",
        "lifetime_value"
    ],
    "justificacion": [
        "Variable objetivo.",
        "Identificador sin significado predictivo generalizable.",
        "Excluida por inconsistencias internas de ubicación.",
        "Fecha original reemplazada por variables derivadas.",
        "Fecha original reemplazada por variables derivadas.",
        "Reemplazada por coupon_code_present_flag.",
        "Redundante con country.",
        "Reemplazada por total_spent_log para evitar duplicar el concepto.",
        (
            "Exclusión preventiva por definición temporal insuficiente; "
            "no constituye fuga demostrada."
        )
    ]
})

print("Dimensiones del dataset para modelado:", df_model.shape)
print("Clientes únicos:", df_model["customer_id"].nunique())
print("Clases disponibles en churn:", sorted(clases_churn))

display(tabla_exclusiones_modelo)

Dimensiones del dataset para modelado: (15000, 40)
Clientes únicos: 15000
Clases disponibles en churn: [0, 1]


,variable_excluida,justificacion
0,churn,Variable objetivo.
1,customer_id,Identificador sin significado predictivo gener...
2,city,Excluida por inconsistencias internas de ubica...
3,signup_date,Fecha original reemplazada por variables deriv...
4,last_purchase_date,Fecha original reemplazada por variables deriv...
5,coupon_code,Reemplazada por coupon_code_present_flag.
6,country_code,Redundante con country.
7,total_spent,Reemplazada por total_spent_log para evitar du...
8,lifetime_value,Exclusión preventiva por definición temporal i...


**Interpretación.** El archivo procesado se carga en una copia independiente y
la variable objetivo conserva únicamente las clases esperadas. La exclusión de
identificadores, fechas originales y representaciones redundantes evita introducir
señales difíciles de generalizar o duplicar información equivalente.

**Utilidad.** La revisión documenta qué información estará disponible para el
primer modelo y permite defender cada exclusión metodológica.

**Limitación.** La disponibilidad temporal real de algunas variables no está
documentada. Por ello, `lifetime_value` queda fuera hasta contar con una definición
operacional más precisa.

### 21.2 Selección de variables predictoras

Las variables se separan explícitamente en numéricas y categóricas. No se utiliza
`total_spent` junto con `total_spent_log`, y las categorías se conservarán sin
imponerles un orden artificial.

In [42]:
numeric_features = [
    "age",
    "is_premium_user",
    "total_visits",
    "avg_session_time",
    "pages_per_session",
    "email_open_rate",
    "email_click_rate",
    "total_spent_log",
    "avg_order_value",
    "discount_used",
    "support_tickets",
    "refund_requested",
    "delivery_delay_days",
    "satisfaction_score",
    "nps_score",
    "marketing_spend_per_user",
    "last_3_month_purchase_freq",
    "coupon_code_present_flag",
    "date_inconsistency_flag",
    "tenure_days",
    "signup_year",
    "last_purchase_year",
    "location_inconsistency_flag",
    "internet_users_pct",
    "gdp_per_capita_usd"
]

categorical_features = [
    "gender",
    "country",
    "acquisition_channel",
    "device_type",
    "subscription_type",
    "payment_method"
]

model_features = (
    numeric_features
    + categorical_features
)

columnas_modelo_faltantes = sorted(
    set(model_features)
    - set(df_model.columns)
)

if columnas_modelo_faltantes:
    raise ValueError(
        "Faltan variables seleccionadas para el modelo: "
        f"{columnas_modelo_faltantes}"
    )

if len(model_features) != len(set(model_features)):
    raise ValueError(
        "La selección de variables contiene columnas repetidas."
    )

X = df_model[model_features].copy()
y = df_model["churn"].copy()

columnas_prohibidas_en_X = {
    "churn",
    "customer_id",
    "city"
}

columnas_prohibidas_detectadas = sorted(
    columnas_prohibidas_en_X
    & set(X.columns)
)

if columnas_prohibidas_detectadas:
    raise ValueError(
        "X contiene columnas que deben quedar excluidas: "
        f"{columnas_prohibidas_detectadas}"
    )

if len(X) != len(y):
    raise ValueError(
        "X e y no contienen la misma cantidad de filas."
    )

if not X.index.equals(y.index):
    raise ValueError(
        "Los índices de X e y no están alineados."
    )

tabla_variables_modelo = pd.DataFrame({
    "tipo_variable": [
        "Numérica",
        "Categórica"
    ],
    "cantidad_columnas": [
        len(numeric_features),
        len(categorical_features)
    ],
    "columnas": [
        ", ".join(numeric_features),
        ", ".join(categorical_features)
    ]
})

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("Columnas predictoras totales:", len(model_features))

display(tabla_variables_modelo)

Dimensiones de X: (15000, 31)
Dimensiones de y: (15000,)
Columnas predictoras totales: 31


,tipo_variable,cantidad_columnas,columnas
0,Numérica,25,"age, is_premium_user, total_visits, avg_sessio..."
1,Categórica,6,"gender, country, acquisition_channel, device_t..."


**Interpretación.** El conjunto predictivo contiene variables demográficas,
comerciales, conductuales, de satisfacción, soporte, calidad interna y contexto
nacional. La variable objetivo, el identificador y la ciudad quedan fuera de `X`.

**Utilidad.** La selección explícita evita que cambios accidentales en el CSV
incorporen nuevas columnas al modelo sin revisión metodológica.

**Limitación.** Los indicadores externos se repiten entre clientes del mismo país
y año. Su presencia aporta contexto, pero no los convierte en atributos
individuales ni permite atribuir causalidad.

### 21.3 Separación entre entrenamiento y prueba

Se reserva el 20 % de los clientes para prueba mediante una partición aleatoria
estratificada. El uso de `stratify=y` conserva aproximadamente la proporción de
cada clase en ambos conjuntos.

Esta partición es admisible porque existe un cliente único por fila, no hay
identificadores repetidos y el objetivo de esta etapa no está definido como un
pronóstico temporal futuro.

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

if len(X_train) != 12000:
    raise ValueError(
        "El conjunto de entrenamiento no contiene 12.000 clientes. "
        f"Cantidad encontrada: {len(X_train)}"
    )

if len(X_test) != 3000:
    raise ValueError(
        "El conjunto de prueba no contiene 3.000 clientes. "
        f"Cantidad encontrada: {len(X_test)}"
    )

indices_compartidos = (
    X_train.index
    .intersection(X_test.index)
)

if not indices_compartidos.empty:
    raise ValueError(
        "Existen índices compartidos entre entrenamiento y prueba."
    )

def resumir_distribucion_clases(
    nombre_conjunto,
    serie_objetivo
):
    conteos = (
        serie_objetivo
        .value_counts()
        .reindex(
            [0, 1],
            fill_value=0
        )
    )

    total_clientes = int(conteos.sum())

    return {
        "conjunto": nombre_conjunto,
        "clientes_totales": total_clientes,
        "clase_0": int(conteos.loc[0]),
        "clase_1": int(conteos.loc[1]),
        "porcentaje_clase_0": (
            conteos.loc[0]
            / total_clientes
            * 100
        ),
        "porcentaje_clase_1": (
            conteos.loc[1]
            / total_clientes
            * 100
        )
    }

tabla_distribucion_split = pd.DataFrame([
    resumir_distribucion_clases(
        "Total",
        y
    ),
    resumir_distribucion_clases(
        "Entrenamiento",
        y_train
    ),
    resumir_distribucion_clases(
        "Prueba",
        y_test
    )
])

diferencia_maxima_distribucion = (
    tabla_distribucion_split[
        "porcentaje_clase_1"
    ].max()
    - tabla_distribucion_split[
        "porcentaje_clase_1"
    ].min()
)

display(
    tabla_distribucion_split.style.format({
        "porcentaje_clase_0": "{:.2f} %",
        "porcentaje_clase_1": "{:.2f} %"
    })
)

print(
    "Índices compartidos:",
    len(indices_compartidos)
)

print(
    "Diferencia máxima en la proporción de clase 1:",
    f"{diferencia_maxima_distribucion:.4f} puntos porcentuales"
)

,conjunto,clientes_totales,clase_0,clase_1,porcentaje_clase_0,porcentaje_clase_1
0,Total,15000,12702,2298,84.68 %,15.32 %
1,Entrenamiento,12000,10162,1838,84.68 %,15.32 %
2,Prueba,3000,2540,460,84.67 %,15.33 %


Índices compartidos: 0
Diferencia máxima en la proporción de clase 1: 0.0167 puntos porcentuales


**Interpretación.** La estratificación conserva el desbalance observado en el
dataset completo y mantiene conjuntos independientes de 12.000 y 3.000 clientes.

**Utilidad.** Todos los modelos posteriores podrán evaluarse sobre exactamente la
misma partición, evitando que una comparación cambie por utilizar clientes
distintos.

**Limitación.** El dataset no especifica un corte temporal entre observación y
predicción. Por ello, esta partición evalúa generalización sobre clientes no
vistos, pero no reproduce de forma explícita un escenario futuro.

### 21.4 Pipeline de preprocesamiento

Las variables numéricas se imputan mediante la mediana y luego se estandarizan.
Las variables categóricas se imputan con la categoría más frecuente y se
transforman mediante codificación One-Hot.

Todas las transformaciones permanecerán dentro de un `ColumnTransformer`. Su
ajuste ocurrirá únicamente cuando el pipeline reciba `X_train`.

In [44]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)

print(
    "Variables numéricas configuradas:",
    len(numeric_features)
)

print(
    "Variables categóricas configuradas:",
    len(categorical_features)
)

print(
    "El preprocesamiento aún no ha sido ajustado."
)

Variables numéricas configuradas: 25
Variables categóricas configuradas: 6
El preprocesamiento aún no ha sido ajustado.


**Interpretación.** La mediana reduce la sensibilidad de la imputación numérica a
valores extremos. `StandardScaler` evita que las diferencias de escala dominen la
regresión logística, mientras que One-Hot representa categorías sin imponer un
orden artificial.

`handle_unknown="ignore"` evita errores si el conjunto de prueba contiene una
categoría no observada durante el entrenamiento.

**Utilidad.** Al integrar las transformaciones en el pipeline se evita ajustar
estadísticos o categorías utilizando información del conjunto de prueba.

**Limitación.** La imputación conserva todas las filas, pero sustituye valores
desconocidos por estimaciones simples que no recuperan la información original.

### 21.5 Baseline de clase mayoritaria

Antes de entrenar un clasificador real se calcula una referencia mínima. El
baseline predice siempre la clase más frecuente observada en entrenamiento.

In [45]:
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

baseline_model.fit(
    X_train,
    y_train
)

y_train_pred_baseline = baseline_model.predict(
    X_train
)

y_test_pred_baseline = baseline_model.predict(
    X_test
)

baseline_accuracy_train = accuracy_score(
    y_train,
    y_train_pred_baseline
)

baseline_accuracy_test = accuracy_score(
    y_test,
    y_test_pred_baseline
)

clase_mayoritaria_entrenamiento = int(
    y_train
    .mode()
    .iloc[0]
)

clases_predichas_baseline = set(
    np.unique(
        y_test_pred_baseline
    ).tolist()
)

if clases_predichas_baseline != {
    clase_mayoritaria_entrenamiento
}:
    raise ValueError(
        "El baseline no está prediciendo únicamente "
        "la clase mayoritaria."
    )

tabla_baseline = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria"
    ],
    "clase_predicha": [
        clase_mayoritaria_entrenamiento
    ],
    "accuracy_entrenamiento_pct": [
        baseline_accuracy_train * 100
    ],
    "accuracy_prueba_pct": [
        baseline_accuracy_test * 100
    ]
})

display(
    tabla_baseline.style.format({
        "accuracy_entrenamiento_pct": "{:.2f} %",
        "accuracy_prueba_pct": "{:.2f} %"
    })
)

print(
    "Clientes churn detectados por el baseline:",
    int(
        np.sum(
            y_test_pred_baseline == 1
        )
    )
)

,modelo,clase_predicha,accuracy_entrenamiento_pct,accuracy_prueba_pct
0,Clase mayoritaria,0,84.68 %,84.67 %


Clientes churn detectados por el baseline: 0


**Interpretación.** La accuracy del baseline es elevada debido al desbalance de
clases, no porque identifique correctamente a los clientes con riesgo de abandono.
Al predecir siempre la clase 0, no detecta clientes churn.

**Utilidad.** Cualquier modelo real debe compararse con esta referencia para
comprobar si aporta una mejora efectiva.

**Limitación.** Un resultado apenas superior al baseline no demuestra utilidad
suficiente para una estrategia de retención.

### 21.6 Primer modelo de clasificación

Se utiliza una regresión logística como primer clasificador. Aunque su nombre
incluye la palabra regresión, corresponde a un algoritmo de clasificación
binaria.

El modelo proporciona una referencia sencilla, funciona con variables numéricas
escaladas y categorías codificadas, y permite comprobar si un clasificador real
supera al baseline.

In [46]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

logistic_pipeline.fit(
    X_train,
    y_train
)

y_train_pred_logistic = logistic_pipeline.predict(
    X_train
)

y_test_pred_logistic = logistic_pipeline.predict(
    X_test
)

print(
    "Pipeline de regresión logística ajustado "
    "únicamente con entrenamiento."
)

print(
    "Predicciones de entrenamiento:",
    len(y_train_pred_logistic)
)

print(
    "Predicciones de prueba:",
    len(y_test_pred_logistic)
)

Pipeline de regresión logística ajustado únicamente con entrenamiento.
Predicciones de entrenamiento: 12000
Predicciones de prueba: 3000


### 21.7 Evaluación mediante accuracy

Se compara la proporción de predicciones correctas en entrenamiento y prueba,
la diferencia entre ambos conjuntos y la mejora frente al baseline de prueba.
Los valores sin redondear se conservan en las variables de cálculo.

In [47]:
logistic_accuracy_train = accuracy_score(
    y_train,
    y_train_pred_logistic
)

logistic_accuracy_test = accuracy_score(
    y_test,
    y_test_pred_logistic
)

train_test_gap = abs(
    logistic_accuracy_train
    - logistic_accuracy_test
)

improvement_over_baseline_test = (
    logistic_accuracy_test
    - baseline_accuracy_test
)

baseline_train_test_gap = abs(
    baseline_accuracy_train
    - baseline_accuracy_test
)

tabla_accuracy = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "Regresión logística"
    ],
    "accuracy_entrenamiento_pct": [
        baseline_accuracy_train * 100,
        logistic_accuracy_train * 100
    ],
    "accuracy_prueba_pct": [
        baseline_accuracy_test * 100,
        logistic_accuracy_test * 100
    ],
    "diferencia_train_test_pp": [
        baseline_train_test_gap * 100,
        train_test_gap * 100
    ],
    "diferencia_vs_baseline_prueba_pp": [
        0.0,
        improvement_over_baseline_test * 100
    ]
})

display(
    tabla_accuracy.style.format({
        "accuracy_entrenamiento_pct": "{:.2f} %",
        "accuracy_prueba_pct": "{:.2f} %",
        "diferencia_train_test_pp": "{:.3f}",
        "diferencia_vs_baseline_prueba_pp": "{:+.3f}"
    })
)

print(
    "Diferencia absoluta train-test:",
    f"{train_test_gap * 100:.3f} puntos porcentuales"
)

print(
    "Diferencia frente al baseline de prueba:",
    (
        f"{improvement_over_baseline_test * 100:+.3f} "
        "puntos porcentuales"
    )
)

if train_test_gap < 0.02:
    print(
        "La diferencia entre entrenamiento y prueba es pequeña; "
        "por accuracy no se observa sobreajuste importante."
    )
else:
    print(
        "La diferencia entre entrenamiento y prueba requiere "
        "revisión antes de descartar sobreajuste."
    )

if abs(improvement_over_baseline_test) < 0.01:
    print(
        "La accuracy de prueba está extremadamente cerca del baseline. "
        "La mejora es marginal y no demuestra utilidad suficiente."
    )
elif improvement_over_baseline_test > 0:
    print(
        "El modelo supera al baseline, pero la utilidad debe revisarse "
        "junto con los errores de clasificación."
    )
else:
    print(
        "El modelo no supera al baseline de clase mayoritaria."
    )

,modelo,accuracy_entrenamiento_pct,accuracy_prueba_pct,diferencia_train_test_pp,diferencia_vs_baseline_prueba_pp
0,Clase mayoritaria,84.68 %,84.67 %,0.017,+0.000
1,Regresión logística,84.84 %,84.80 %,0.042,+0.133


Diferencia absoluta train-test: 0.042 puntos porcentuales
Diferencia frente al baseline de prueba: +0.133 puntos porcentuales
La diferencia entre entrenamiento y prueba es pequeña; por accuracy no se observa sobreajuste importante.
La accuracy de prueba está extremadamente cerca del baseline. La mejora es marginal y no demuestra utilidad suficiente.


**Interpretación.** La estabilidad entre entrenamiento y prueba debe distinguirse
de la utilidad práctica. Una diferencia pequeña entre ambos conjuntos no compensa
una accuracy de prueba cercana a la obtenida al predecir siempre la clase
mayoritaria.

**Utilidad.** La tabla permite evaluar simultáneamente generalización y valor
añadido respecto de una referencia mínima.

**Limitación.** Accuracy resume la proporción total de aciertos, pero puede ocultar
un desempeño deficiente sobre la clase minoritaria.

### 21.8 Matriz de confusión

La matriz de confusión descompone los aciertos y errores según el valor real y el
valor predicho. Esto permite observar cuántos clientes churn fueron detectados y
cuántos quedaron omitidos.

In [48]:
cm_logistic = confusion_matrix(
    y_test,
    y_test_pred_logistic,
    labels=[0, 1]
)

tn, fp, fn, tp = cm_logistic.ravel()

tabla_matriz_confusion = pd.DataFrame(
    cm_logistic,
    index=pd.Index(
        [
            "No churn",
            "Churn"
        ],
        name="Valor real"
    ),
    columns=pd.Index(
        [
            "No churn",
            "Churn"
        ],
        name="Valor predicho"
    )
)

total_real_churn = int(
    fn + tp
)

porcentaje_churn_detectado = (
    tp
    / total_real_churn
    * 100
)

porcentaje_churn_omitido = (
    fn
    / total_real_churn
    * 100
)

resumen_matriz_confusion = pd.DataFrame({
    "resultado": [
        "No churn correctamente clasificados",
        "No churn clasificados como churn",
        "Churn omitidos",
        "Churn correctamente detectados",
        "Total real de churn",
        "Porcentaje de churn detectado",
        "Porcentaje de churn omitido"
    ],
    "valor": [
        int(tn),
        int(fp),
        int(fn),
        int(tp),
        total_real_churn,
        porcentaje_churn_detectado,
        porcentaje_churn_omitido
    ],
    "unidad": [
        "clientes",
        "clientes",
        "clientes",
        "clientes",
        "clientes",
        "%",
        "%"
    ]
})

display(tabla_matriz_confusion)
display(resumen_matriz_confusion)

fig_matriz_confusion = px.imshow(
    tabla_matriz_confusion,
    text_auto=True,
    aspect="auto",
    labels={
        "x": "Valor predicho",
        "y": "Valor real",
        "color": "Clientes"
    },
    title="Matriz de confusión de la regresión logística"
)

fig_matriz_confusion.show()

print(
    "Clientes churn detectados:",
    f"{int(tp)} de {total_real_churn} "
    f"({porcentaje_churn_detectado:.2f} %)"
)

print(
    "Clientes churn omitidos:",
    f"{int(fn)} de {total_real_churn} "
    f"({porcentaje_churn_omitido:.2f} %)"
)

if fn > tp:
    print(
        "La accuracy global oculta que el modelo omite "
        "más clientes churn de los que detecta."
    )
else:
    print(
        "El modelo detecta al menos tantos clientes churn "
        "como los que omite; el resultado debe revisarse "
        "junto con la accuracy y el baseline."
    )

Valor predicho,No churn,Churn
Valor real,,
No churn,2455,85
Churn,371,89


,resultado,valor,unidad
0,No churn correctamente clasificados,"2,455.00",clientes
1,No churn clasificados como churn,85.00,clientes
2,Churn omitidos,371.00,clientes
3,Churn correctamente detectados,89.00,clientes
4,Total real de churn,460.00,clientes
5,Porcentaje de churn detectado,19.35,%
6,Porcentaje de churn omitido,80.65,%


Clientes churn detectados: 89 de 460 (19.35 %)
Clientes churn omitidos: 371 de 460 (80.65 %)
La accuracy global oculta que el modelo omite más clientes churn de los que detecta.


**Interpretación.** La matriz permite verificar si la accuracy total se sostiene
por la clasificación de la clase mayoritaria. El primer modelo no puede
considerarse suficiente para retención cuando omite una cantidad elevada de
clientes churn, aunque la diferencia entre entrenamiento y prueba sea estable.

**Utilidad.** Los clientes correctamente detectados representan casos que podrían
priorizarse para acciones preventivas. Los abandonos omitidos muestran el costo
operativo de depender únicamente de este modelo.

**Limitación.** La matriz utiliza el umbral predeterminado del clasificador y esta
etapa no modifica umbrales, pesos ni distribución de clases.

### 21.9 Conclusión y limitaciones del modelo inicial

El pipeline ajusta imputación, escalamiento y codificación únicamente con
`X_train`, por lo que el conjunto de prueba no interviene en la preparación del
modelo. La partición estratificada conserva el desbalance de la variable objetivo.

El baseline alcanza una accuracy elevada porque predice siempre la clase
mayoritaria. La regresión logística debe interpretarse frente a esa referencia:
una mejora marginal y una diferencia train-test pequeña indican estabilidad, pero
no una solución fuerte. La matriz de confusión demuestra que la accuracy por sí
sola puede ocultar una cantidad alta de clientes churn omitidos.

Este primer clasificador no debe seleccionarse todavía como modelo final.
Posteriormente se comparará con otro clasificador utilizando exactamente el mismo
split. `y_test` no se modificará ni se utilizará para ajustar hiperparámetros o
seleccionar transformaciones.

Las principales limitaciones son:

- no existe un corte temporal explícito entre observación y predicción;
- `lifetime_value` se excluyó preventivamente por ambigüedad temporal;
- los indicadores externos son valores nacionales repetidos entre clientes;
- accuracy no describe por sí sola la detección de la clase minoritaria;
- no se aplicó balanceo ni ajuste de pesos;
- no se realizó optimización;
- solo se evaluó un primer modelo de referencia.

## 22. Comparación de modelos de clasificación

Se amplía la evaluación para contrastar tres enfoques de clasificación bajo exactamente las mismas condiciones experimentales:

- regresión logística como referencia lineal;
- árbol de decisión como modelo no lineal basado en reglas;
- bosque aleatorio como método ensamblado que combina múltiples árboles.

Los tres candidatos reutilizan las mismas variables, la misma partición estratificada y el mismo pipeline de preprocesamiento. De esta forma, las diferencias observadas se atribuyen al comportamiento de los algoritmos y no a cambios en los datos.

La selección no se basará únicamente en una diferencia marginal de accuracy. También se revisarán la estabilidad entre entrenamiento y prueba, las matrices de confusión y la cantidad de clientes churn correctamente identificados y omitidos.

### 22.1 Modelos no lineales adicionales

El árbol de decisión puede representar relaciones no lineales e interacciones entre variables mediante reglas interpretables. Su principal limitación es que, sin restricciones, puede adaptarse excesivamente al conjunto de entrenamiento.

El bosque aleatorio combina múltiples árboles construidos sobre muestras y subconjuntos de variables. Esta estrategia suele reducir la dependencia de un único árbol y ofrece una referencia ensamblada más robusta, aunque disminuye la interpretación directa de las reglas.

Ambos modelos se utilizan con su configuración base, una semilla fija y sin optimización de hiperparámetros. El objetivo es comparar comportamientos iniciales, no buscar todavía la mejor configuración posible.

In [49]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)

forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

tree_pipeline.fit(
    X_train,
    y_train
)

forest_pipeline.fit(
    X_train,
    y_train
)

y_train_pred_tree = tree_pipeline.predict(
    X_train
)

y_test_pred_tree = tree_pipeline.predict(
    X_test
)

y_train_pred_forest = forest_pipeline.predict(
    X_train
)

y_test_pred_forest = forest_pipeline.predict(
    X_test
)

print(
    "Modelos no lineales ajustados únicamente "
    "con el conjunto de entrenamiento."
)

print(
    "Predicciones del árbol:",
    len(y_train_pred_tree),
    "en entrenamiento y",
    len(y_test_pred_tree),
    "en prueba."
)

print(
    "Predicciones del bosque aleatorio:",
    len(y_train_pred_forest),
    "en entrenamiento y",
    len(y_test_pred_forest),
    "en prueba."
)

Modelos no lineales ajustados únicamente con el conjunto de entrenamiento.
Predicciones del árbol: 12000 en entrenamiento y 3000 en prueba.
Predicciones del bosque aleatorio: 12000 en entrenamiento y 3000 en prueba.


**Interpretación.** Los tres clasificadores reciben las mismas columnas y las mismas observaciones. El uso de `clone` evita compartir entre modelos un preprocesador previamente ajustado.

**Utilidad.** La comparación permite contrastar un límite lineal, reglas no lineales individuales y un ensamble de árboles.

**Limitación.** El árbol y el bosque aleatorio pueden alcanzar una accuracy de entrenamiento muy alta. Ese resultado no implica mejor generalización y debe compararse con el desempeño en prueba.

### 22.2 Comparación de accuracy y estabilidad

Se calculan las accuracies de entrenamiento y prueba, la diferencia absoluta entre ambas y el cambio frente al baseline de clase mayoritaria.

Una accuracy de entrenamiento muy superior a la de prueba se considera una señal de posible sobreajuste. Además, un modelo que no supera el baseline no aporta una mejora suficiente bajo la métrica académica principal.

In [50]:
tree_accuracy_train = accuracy_score(
    y_train,
    y_train_pred_tree
)

tree_accuracy_test = accuracy_score(
    y_test,
    y_test_pred_tree
)

forest_accuracy_train = accuracy_score(
    y_train,
    y_train_pred_forest
)

forest_accuracy_test = accuracy_score(
    y_test,
    y_test_pred_forest
)

tree_train_test_gap = abs(
    tree_accuracy_train
    - tree_accuracy_test
)

forest_train_test_gap = abs(
    forest_accuracy_train
    - forest_accuracy_test
)

tree_improvement_over_baseline_test = (
    tree_accuracy_test
    - baseline_accuracy_test
)

forest_improvement_over_baseline_test = (
    forest_accuracy_test
    - baseline_accuracy_test
)

tabla_comparacion_accuracy = pd.DataFrame({
    "modelo": [
        "Clase mayoritaria",
        "Regresión logística",
        "Árbol de decisión",
        "Bosque aleatorio"
    ],
    "accuracy_entrenamiento": [
        baseline_accuracy_train,
        logistic_accuracy_train,
        tree_accuracy_train,
        forest_accuracy_train
    ],
    "accuracy_prueba": [
        baseline_accuracy_test,
        logistic_accuracy_test,
        tree_accuracy_test,
        forest_accuracy_test
    ],
    "diferencia_train_test": [
        baseline_train_test_gap,
        train_test_gap,
        tree_train_test_gap,
        forest_train_test_gap
    ],
    "diferencia_vs_baseline_prueba": [
        0.0,
        improvement_over_baseline_test,
        tree_improvement_over_baseline_test,
        forest_improvement_over_baseline_test
    ]
})

tabla_comparacion_accuracy_mostrar = (
    tabla_comparacion_accuracy
    .assign(
        accuracy_entrenamiento_pct=lambda df: (
            df["accuracy_entrenamiento"] * 100
        ),
        accuracy_prueba_pct=lambda df: (
            df["accuracy_prueba"] * 100
        ),
        diferencia_train_test_pp=lambda df: (
            df["diferencia_train_test"] * 100
        ),
        diferencia_vs_baseline_prueba_pp=lambda df: (
            df["diferencia_vs_baseline_prueba"] * 100
        )
    )
    [
        [
            "modelo",
            "accuracy_entrenamiento_pct",
            "accuracy_prueba_pct",
            "diferencia_train_test_pp",
            "diferencia_vs_baseline_prueba_pp"
        ]
    ]
)

display(
    tabla_comparacion_accuracy_mostrar.style.format({
        "accuracy_entrenamiento_pct": "{:.2f} %",
        "accuracy_prueba_pct": "{:.2f} %",
        "diferencia_train_test_pp": "{:.3f}",
        "diferencia_vs_baseline_prueba_pp": "{:+.3f}"
    })
)

tabla_accuracy_grafico = (
    tabla_comparacion_accuracy
    .melt(
        id_vars="modelo",
        value_vars=[
            "accuracy_entrenamiento",
            "accuracy_prueba"
        ],
        var_name="conjunto",
        value_name="accuracy"
    )
)

tabla_accuracy_grafico["accuracy_pct"] = (
    tabla_accuracy_grafico["accuracy"] * 100
)

tabla_accuracy_grafico["conjunto"] = (
    tabla_accuracy_grafico["conjunto"]
    .map({
        "accuracy_entrenamiento": "Entrenamiento",
        "accuracy_prueba": "Prueba"
    })
)

fig_accuracy_modelos = px.bar(
    tabla_accuracy_grafico,
    x="modelo",
    y="accuracy_pct",
    color="conjunto",
    barmode="group",
    text_auto=".2f",
    labels={
        "modelo": "Modelo",
        "accuracy_pct": "Accuracy (%)",
        "conjunto": "Conjunto"
    },
    title="Accuracy de entrenamiento y prueba por modelo"
)

fig_accuracy_modelos.update_layout(
    yaxis_range=[0, 100]
)

fig_accuracy_modelos.show()

for fila in tabla_comparacion_accuracy_mostrar.itertuples(index=False):
    print(
        f"{fila.modelo}: accuracy de prueba "
        f"{fila.accuracy_prueba_pct:.2f} %, "
        f"brecha train-test "
        f"{fila.diferencia_train_test_pp:.3f} puntos porcentuales."
    )

,modelo,accuracy_entrenamiento_pct,accuracy_prueba_pct,diferencia_train_test_pp,diferencia_vs_baseline_prueba_pp
0,Clase mayoritaria,84.68 %,84.67 %,0.017,+0.000
1,Regresión logística,84.84 %,84.80 %,0.042,+0.133
2,Árbol de decisión,100.00 %,86.17 %,13.833,+1.500
3,Bosque aleatorio,100.00 %,84.87 %,15.133,+0.200


Clase mayoritaria: accuracy de prueba 84.67 %, brecha train-test 0.017 puntos porcentuales.
Regresión logística: accuracy de prueba 84.80 %, brecha train-test 0.042 puntos porcentuales.
Árbol de decisión: accuracy de prueba 86.17 %, brecha train-test 13.833 puntos porcentuales.
Bosque aleatorio: accuracy de prueba 84.87 %, brecha train-test 15.133 puntos porcentuales.


**Interpretación.** La tabla y el gráfico diferencian rendimiento aparente y capacidad de generalización. Un modelo puede ajustarse casi perfectamente al entrenamiento y aun así obtener una accuracy inferior en clientes no utilizados durante el ajuste.

**Utilidad.** Comparar ambos conjuntos evita seleccionar automáticamente el algoritmo con mayor resultado de entrenamiento.

**Limitación.** Accuracy sigue dominada por la clase mayoritaria. Por ello, la selección se complementa con los errores específicos observados en las matrices de confusión.

### 22.3 Comparación de matrices de confusión

Se calculan las matrices de confusión de los tres clasificadores con el mismo orden de clases. Esto permite distinguir si una variación en accuracy proviene de clasificar mejor a la mayoría de clientes sin abandono o de reconocer una mayor cantidad de abandonos reales.

In [51]:
tn_logistic, fp_logistic, fn_logistic, tp_logistic = (
    cm_logistic.ravel()
)

cm_tree = confusion_matrix(
    y_test,
    y_test_pred_tree,
    labels=[0, 1]
)

cm_forest = confusion_matrix(
    y_test,
    y_test_pred_forest,
    labels=[0, 1]
)

tn_tree, fp_tree, fn_tree, tp_tree = cm_tree.ravel()
tn_forest, fp_forest, fn_forest, tp_forest = cm_forest.ravel()

etiquetas_clase = [
    "No churn",
    "Churn"
]

matrices_modelos = {
    "Regresión logística": cm_logistic,
    "Árbol de decisión": cm_tree,
    "Bosque aleatorio": cm_forest
}

tablas_matrices = {}

for nombre_modelo, matriz in matrices_modelos.items():
    tabla_matriz = pd.DataFrame(
        matriz,
        index=pd.Index(
            etiquetas_clase,
            name="Valor real"
        ),
        columns=pd.Index(
            etiquetas_clase,
            name="Valor predicho"
        )
    )

    tablas_matrices[nombre_modelo] = tabla_matriz

    print(nombre_modelo)
    display(tabla_matriz)

    figura_matriz = px.imshow(
        tabla_matriz,
        text_auto=True,
        aspect="auto",
        labels={
            "x": "Valor predicho",
            "y": "Valor real",
            "color": "Clientes"
        },
        title=f"Matriz de confusión: {nombre_modelo.lower()}"
    )

    figura_matriz.show()

Regresión logística


Valor predicho,No churn,Churn
Valor real,,
No churn,2455,85
Churn,371,89


Árbol de decisión


Valor predicho,No churn,Churn
Valor real,,
No churn,2352,188
Churn,227,233


Bosque aleatorio


Valor predicho,No churn,Churn
Valor real,,
No churn,2456,84
Churn,370,90


**Interpretación.** Las matrices muestran que dos modelos con accuracies similares pueden distribuir sus errores de forma distinta. La clasificación de churn debe revisarse explícitamente porque la mayoría de las observaciones pertenece a la clase no churn.

**Utilidad.** Esta lectura permite considerar la capacidad de detectar clientes en riesgo y no solo el total de predicciones correctas.

**Limitación.** Las matrices corresponden a un único conjunto de prueba. No garantizan el mismo comportamiento ante cambios futuros en la población o en el proceso comercial.

### 22.4 Clientes churn detectados y omitidos

La clase minoritaria se resume mediante cantidades y porcentajes derivados directamente de cada matriz. No se introducen métricas adicionales en esta etapa; la comparación mantiene accuracy como medida académica principal y utiliza los conteos para interpretar su efecto sobre churn.

In [52]:
total_churn_prueba = int(
    (y_test == 1).sum()
)

resultados_matrices = {
    "Regresión logística": {
        "tn": int(tn_logistic),
        "fp": int(fp_logistic),
        "fn": int(fn_logistic),
        "tp": int(tp_logistic)
    },
    "Árbol de decisión": {
        "tn": int(tn_tree),
        "fp": int(fp_tree),
        "fn": int(fn_tree),
        "tp": int(tp_tree)
    },
    "Bosque aleatorio": {
        "tn": int(tn_forest),
        "fp": int(fp_forest),
        "fn": int(fn_forest),
        "tp": int(tp_forest)
    }
}

for nombre_modelo, valores in resultados_matrices.items():
    if total_churn_prueba != valores["fn"] + valores["tp"]:
        raise ValueError(
            f"La matriz de {nombre_modelo} no coincide "
            "con el total real de churn."
        )

tabla_deteccion_churn = pd.DataFrame([
    {
        "modelo": nombre_modelo,
        "churn_reales": total_churn_prueba,
        "churn_detectados": valores["tp"],
        "churn_omitidos": valores["fn"],
        "porcentaje_churn_detectado": (
            valores["tp"] / total_churn_prueba * 100
        ),
        "porcentaje_churn_omitido": (
            valores["fn"] / total_churn_prueba * 100
        ),
        "no_churn_clasificados_como_churn": valores["fp"]
    }
    for nombre_modelo, valores in resultados_matrices.items()
])

display(
    tabla_deteccion_churn.style.format({
        "porcentaje_churn_detectado": "{:.2f} %",
        "porcentaje_churn_omitido": "{:.2f} %"
    })
)

tabla_churn_grafico = (
    tabla_deteccion_churn
    .melt(
        id_vars="modelo",
        value_vars=[
            "churn_detectados",
            "churn_omitidos"
        ],
        var_name="resultado",
        value_name="clientes"
    )
)

tabla_churn_grafico["resultado"] = (
    tabla_churn_grafico["resultado"]
    .map({
        "churn_detectados": "Detectados",
        "churn_omitidos": "Omitidos"
    })
)

fig_churn_modelos = px.bar(
    tabla_churn_grafico,
    x="modelo",
    y="clientes",
    color="resultado",
    barmode="group",
    text_auto=True,
    labels={
        "modelo": "Modelo",
        "clientes": "Clientes churn",
        "resultado": "Resultado"
    },
    title="Clientes churn detectados y omitidos por modelo"
)

fig_churn_modelos.show()

for fila in tabla_deteccion_churn.itertuples(index=False):
    print(
        f"{fila.modelo}: detecta {fila.churn_detectados} de "
        f"{fila.churn_reales} clientes churn y omite "
        f"{fila.churn_omitidos}."
    )

,modelo,churn_reales,churn_detectados,churn_omitidos,porcentaje_churn_detectado,porcentaje_churn_omitido,no_churn_clasificados_como_churn
0,Regresión logística,460,89,371,19.35 %,80.65 %,85
1,Árbol de decisión,460,233,227,50.65 %,49.35 %,188
2,Bosque aleatorio,460,90,370,19.57 %,80.43 %,84


Regresión logística: detecta 89 de 460 clientes churn y omite 371.
Árbol de decisión: detecta 233 de 460 clientes churn y omite 227.
Bosque aleatorio: detecta 90 de 460 clientes churn y omite 370.


**Interpretación.** La cantidad de churn detectados representa una dimensión de utilidad comercial distinta de la accuracy global. Un modelo también puede aumentar la detección a costa de clasificar como riesgosos a más clientes que realmente no abandonan.

**Utilidad.** El resumen permite revisar simultáneamente abandonos detectados, abandonos omitidos y posibles intervenciones innecesarias.

**Limitación.** El dataset no informa el costo económico de cada tipo de error. Por ello, los conteos no pueden transformarse todavía en una decisión financiera definitiva.

### 22.5 Selección razonada del modelo

La selección utiliza una regla explícita y reproducible:

1. el candidato debe superar el baseline en prueba;
2. una diferencia train-test igual o superior a cinco puntos porcentuales se considera una señal importante de inestabilidad;
3. entre candidatos viables cuyas accuracies de prueba estén separadas por un máximo de medio punto porcentual, se prioriza el que detecta más clientes churn;
4. si la diferencia de accuracy supera ese margen, se conserva el candidato con mejor accuracy de prueba y brecha aceptable.

Esta regla no optimiza hiperparámetros ni altera las predicciones. Solo formaliza la interpretación de los resultados obtenidos bajo el mismo experimento.

In [53]:
resultados_candidatos = pd.DataFrame({
    "modelo": [
        "Regresión logística",
        "Árbol de decisión",
        "Bosque aleatorio"
    ],
    "accuracy_entrenamiento": [
        logistic_accuracy_train,
        tree_accuracy_train,
        forest_accuracy_train
    ],
    "accuracy_prueba": [
        logistic_accuracy_test,
        tree_accuracy_test,
        forest_accuracy_test
    ],
    "diferencia_train_test": [
        train_test_gap,
        tree_train_test_gap,
        forest_train_test_gap
    ],
    "diferencia_vs_baseline_prueba": [
        improvement_over_baseline_test,
        tree_improvement_over_baseline_test,
        forest_improvement_over_baseline_test
    ],
    "churn_detectados": [
        int(tp_logistic),
        int(tp_tree),
        int(tp_forest)
    ],
    "churn_omitidos": [
        int(fn_logistic),
        int(fn_tree),
        int(fn_forest)
    ],
    "no_churn_clasificados_como_churn": [
        int(fp_logistic),
        int(fp_tree),
        int(fp_forest)
    ]
})

resultados_candidatos["supera_baseline"] = (
    resultados_candidatos["accuracy_prueba"]
    > baseline_accuracy_test
)

resultados_candidatos["brecha_aceptable"] = (
    resultados_candidatos["diferencia_train_test"]
    < 0.05
)

candidatos_viables = resultados_candidatos.loc[
    resultados_candidatos["supera_baseline"]
    & resultados_candidatos["brecha_aceptable"]
].copy()

if candidatos_viables.empty:
    modelo_seleccionado = None

    print(
        "Ningún candidato supera simultáneamente el baseline "
        "y el criterio de estabilidad."
    )

    print(
        "No se selecciona un modelo definitivo con la evidencia actual."
    )
else:
    mejor_accuracy_viable = (
        candidatos_viables["accuracy_prueba"]
        .max()
    )

    candidatos_accuracy_similar = candidatos_viables.loc[
        candidatos_viables["accuracy_prueba"]
        >= mejor_accuracy_viable - 0.005
    ].copy()

    fila_seleccionada = (
        candidatos_accuracy_similar
        .sort_values(
            by=[
                "churn_detectados",
                "diferencia_train_test",
                "accuracy_prueba"
            ],
            ascending=[
                False,
                True,
                False
            ]
        )
        .iloc[0]
    )

    modelo_seleccionado = fila_seleccionada["modelo"]

    print(
        "Modelo candidato seleccionado para la etapa siguiente:",
        modelo_seleccionado
    )

    print(
        "Accuracy de prueba:",
        f"{fila_seleccionada['accuracy_prueba'] * 100:.2f} %"
    )

    print(
        "Diferencia train-test:",
        (
            f"{fila_seleccionada['diferencia_train_test'] * 100:.3f} "
            "puntos porcentuales"
        )
    )

    print(
        "Mejora frente al baseline:",
        (
            f"{fila_seleccionada['diferencia_vs_baseline_prueba'] * 100:+.3f} "
            "puntos porcentuales"
        )
    )

    print(
        "Clientes churn detectados:",
        int(fila_seleccionada["churn_detectados"])
    )

    print(
        "Clientes churn omitidos:",
        int(fila_seleccionada["churn_omitidos"])
    )

tabla_seleccion_modelos = (
    resultados_candidatos
    .assign(
        accuracy_entrenamiento_pct=lambda df: (
            df["accuracy_entrenamiento"] * 100
        ),
        accuracy_prueba_pct=lambda df: (
            df["accuracy_prueba"] * 100
        ),
        diferencia_train_test_pp=lambda df: (
            df["diferencia_train_test"] * 100
        ),
        diferencia_vs_baseline_prueba_pp=lambda df: (
            df["diferencia_vs_baseline_prueba"] * 100
        )
    )
    [
        [
            "modelo",
            "accuracy_entrenamiento_pct",
            "accuracy_prueba_pct",
            "diferencia_train_test_pp",
            "diferencia_vs_baseline_prueba_pp",
            "churn_detectados",
            "churn_omitidos",
            "no_churn_clasificados_como_churn",
            "supera_baseline",
            "brecha_aceptable"
        ]
    ]
)

display(
    tabla_seleccion_modelos.style.format({
        "accuracy_entrenamiento_pct": "{:.2f} %",
        "accuracy_prueba_pct": "{:.2f} %",
        "diferencia_train_test_pp": "{:.3f}",
        "diferencia_vs_baseline_prueba_pp": "{:+.3f}"
    })
)

Modelo candidato seleccionado para la etapa siguiente: Regresión logística
Accuracy de prueba: 84.80 %
Diferencia train-test: 0.042 puntos porcentuales
Mejora frente al baseline: +0.133 puntos porcentuales
Clientes churn detectados: 89
Clientes churn omitidos: 371


,modelo,accuracy_entrenamiento_pct,accuracy_prueba_pct,diferencia_train_test_pp,diferencia_vs_baseline_prueba_pp,churn_detectados,churn_omitidos,no_churn_clasificados_como_churn,supera_baseline,brecha_aceptable
0,Regresión logística,84.84 %,84.80 %,0.042,+0.133,89,371,85,True,True
1,Árbol de decisión,100.00 %,86.17 %,13.833,+1.500,233,227,188,True,False
2,Bosque aleatorio,100.00 %,84.87 %,15.133,+0.200,90,370,84,True,False


### 22.6 Conclusión y limitaciones de la comparación

La comparación mantiene constantes las variables, el preprocesamiento y la partición. Las diferencias observadas corresponden, por tanto, al comportamiento de una regresión logística, un árbol de decisión y un bosque aleatorio bajo las mismas condiciones.

El modelo seleccionado por la regla anterior debe entenderse como el candidato más adecuado entre los algoritmos evaluados, no como una demostración de rendimiento óptimo. La regresión logística aporta una referencia estable e interpretable; el árbol puede representar relaciones no lineales, pero es sensible al sobreajuste; el bosque aleatorio reduce la dependencia de un único árbol, aunque también puede mostrar una brecha elevada y es menos interpretable.

La accuracy conserva las restricciones derivadas del desbalance. Las matrices de confusión y los conteos permiten observar el comportamiento sobre churn, pero el dataset no proporciona costos económicos ni un corte temporal explícito. Tampoco se aplicaron balanceo, ajuste de pesos, validación cruzada u optimización de hiperparámetros.

En la etapa siguiente deben utilizarse únicamente los resultados reales del candidato seleccionado. El conjunto de prueba no debe modificarse ni utilizarse para ajustar hiperparámetros a partir de predicciones individuales.

## 23. Panel interactivo consolidado

Se construye una visualización consolidada con Plotly para comunicar los principales hallazgos del dataset y del modelado. El archivo resultante funciona directamente en un navegador, sin servidor ni dependencias de ejecución adicionales.

El dashboard incorpora:

- cantidad de clientes;
- tasa de churn;
- gasto promedio;
- satisfacción promedio;
- comparación de churn por país;
- comparación por canal de adquisición;
- resultados globales de los modelos;
- un selector de país que actualiza los indicadores y las vistas descriptivas.

Los resultados del modelo corresponden al conjunto de prueba global. El selector segmenta el análisis descriptivo, pero no vuelve a entrenar ni evalúa los modelos por país.

### 23.1 Preparación y validación de los datos

Se carga una copia independiente del dataset procesado y se verifican las columnas necesarias para los indicadores y visualizaciones. Las categorías faltantes se representan como `Sin información` únicamente para el agrupamiento visual; no se modifican los archivos de datos.

In [54]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

columnas_dashboard = [
    "customer_id",
    "churn",
    "country",
    "acquisition_channel",
    "total_spent",
    "satisfaction_score"
]

df_dashboard = pd.read_csv(
    ruta_dataset_procesado
)

columnas_faltantes_dashboard = sorted(
    set(columnas_dashboard)
    - set(df_dashboard.columns)
)

if columnas_faltantes_dashboard:
    raise ValueError(
        "Faltan columnas necesarias para el dashboard: "
        f"{columnas_faltantes_dashboard}"
    )

if df_dashboard.empty:
    raise ValueError(
        "El dataset procesado está vacío."
    )

if df_dashboard["customer_id"].duplicated().any():
    raise ValueError(
        "customer_id contiene duplicados en el dataset del dashboard."
    )

if df_dashboard["churn"].isna().any():
    raise ValueError(
        "churn contiene valores faltantes."
    )

clases_dashboard = set(
    df_dashboard["churn"].unique().tolist()
)

if not clases_dashboard.issubset({0, 1}):
    raise ValueError(
        "churn debe contener únicamente las clases 0 y 1."
    )

df_dashboard["country_dashboard"] = (
    df_dashboard["country"]
    .fillna("Sin información")
    .astype(str)
)

df_dashboard["acquisition_channel_dashboard"] = (
    df_dashboard["acquisition_channel"]
    .fillna("Sin información")
    .astype(str)
)

print("Dataset preparado para el dashboard.")
print("Dimensiones:", df_dashboard.shape)
print(
    "Países disponibles:",
    sorted(df_dashboard["country_dashboard"].unique().tolist())
)
print(
    "Canales disponibles:",
    sorted(
        df_dashboard[
            "acquisition_channel_dashboard"
        ].unique().tolist()
    )
)

Dataset preparado para el dashboard.
Dimensiones: (15000, 42)
Países disponibles: ['Bangladesh', 'Germany', 'India', 'UK', 'USA']
Canales disponibles: ['Email', 'Facebook Ads', 'Google Ads', 'Organic', 'Referral']


**Interpretación.** La validación garantiza que cada fila siga representando un cliente único y que los indicadores se calculen sobre una variable objetivo binaria válida.

**Utilidad.** Las categorías sin información permanecen visibles como un grupo explícito, evitando que desaparezcan silenciosamente de los gráficos.

**Limitación.** La segmentación describe asociaciones observadas; no establece que un país o canal cause el abandono.

### 23.2 Tablas para indicadores y visualizaciones

Se preparan funciones que calculan los cuatro indicadores, las tasas de churn por país y canal, y el resumen global de los modelos. Todas las tasas se calculan a partir del subconjunto visible y no se escriben como constantes.

In [55]:
def calcular_kpis_dashboard(datos):
    total_clientes = int(len(datos))

    if total_clientes == 0:
        return {
            "clientes": 0,
            "tasa_churn_pct": 0.0,
            "gasto_promedio": None,
            "satisfaccion_promedio": None
        }

    return {
        "clientes": total_clientes,
        "tasa_churn_pct": float(
            datos["churn"].mean() * 100
        ),
        "gasto_promedio": (
            float(datos["total_spent"].mean())
            if datos["total_spent"].notna().any()
            else None
        ),
        "satisfaccion_promedio": (
            float(datos["satisfaction_score"].mean())
            if datos["satisfaction_score"].notna().any()
            else None
        )
    }


def calcular_tasa_por_categoria(datos, columna):
    if datos.empty:
        return pd.DataFrame({
            columna: ["Sin observaciones"],
            "clientes": [0],
            "clientes_churn": [0],
            "tasa_churn_pct": [0.0]
        })

    tabla = (
        datos
        .groupby(
            columna,
            dropna=False,
            as_index=False
        )
        .agg(
            clientes=("customer_id", "size"),
            clientes_churn=("churn", "sum")
        )
    )

    tabla["tasa_churn_pct"] = (
        tabla["clientes_churn"]
        / tabla["clientes"]
        * 100
    )

    return tabla.sort_values(
        "tasa_churn_pct",
        ascending=False
    )


if total_churn_prueba <= 0:
    raise ValueError(
        "El conjunto de prueba no contiene clientes churn."
    )

tabla_modelos_dashboard = (
    resultados_candidatos
    .assign(
        accuracy_prueba_pct=lambda df: (
            df["accuracy_prueba"] * 100
        ),
        churn_detectado_pct=lambda df: (
            df["churn_detectados"]
            / total_churn_prueba
            * 100
        )
    )
    [
        [
            "modelo",
            "accuracy_prueba_pct",
            "churn_detectado_pct",
            "churn_detectados",
            "churn_omitidos"
        ]
    ]
)

kpis_globales = calcular_kpis_dashboard(
    df_dashboard
)

tabla_paises_global = calcular_tasa_por_categoria(
    df_dashboard,
    "country_dashboard"
)

tabla_canales_global = calcular_tasa_por_categoria(
    df_dashboard,
    "acquisition_channel_dashboard"
)

print("Indicadores globales calculados:")
display(pd.DataFrame([kpis_globales]))

print("Resumen global de modelos:")
display(tabla_modelos_dashboard)

Indicadores globales calculados:


,clientes,tasa_churn_pct,gasto_promedio,satisfaccion_promedio
0,15000,15.32,524.36,3.59


Resumen global de modelos:


,modelo,accuracy_prueba_pct,churn_detectado_pct,churn_detectados,churn_omitidos
0,Regresión logística,84.80,19.35,89,371
1,Árbol de decisión,86.17,50.65,233,227
2,Bosque aleatorio,84.87,19.57,90,370


### 23.3 Construcción del dashboard Plotly

La figura utiliza subgráficos para integrar indicadores y comparaciones complementarias. El selector permite observar el total general o un país específico.

La comparación de modelos permanece global porque el split y la evaluación se realizaron para el conjunto completo. No se presentan métricas segmentadas que no hayan sido calculadas mediante una evaluación metodológicamente separada.

In [56]:
opciones_pais = [
    "Todos los países"
] + sorted(
    df_dashboard[
        "country_dashboard"
    ].unique().tolist()
)

fig_dashboard = make_subplots(
    rows=3,
    cols=4,
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"}
        ],
        [
            {"type": "xy", "colspan": 2},
            None,
            {"type": "xy", "colspan": 2},
            None
        ],
        [
            {"type": "xy", "colspan": 4},
            None,
            None,
            None
        ]
    ],
    subplot_titles=(
        "",
        "",
        "",
        "",
        "Tasa de churn por país",
        "Tasa de churn por canal de adquisición",
        "Rendimiento global de los modelos"
    ),
    vertical_spacing=0.13,
    horizontal_spacing=0.08
)

trazas_por_opcion = 8

for indice_opcion, opcion_pais in enumerate(opciones_pais):
    if opcion_pais == "Todos los países":
        datos_segmento = df_dashboard.copy()
    else:
        datos_segmento = df_dashboard.loc[
            df_dashboard["country_dashboard"]
            == opcion_pais
        ].copy()

    kpis_segmento = calcular_kpis_dashboard(
        datos_segmento
    )

    tabla_paises_segmento = calcular_tasa_por_categoria(
        datos_segmento,
        "country_dashboard"
    )

    tabla_canales_segmento = calcular_tasa_por_categoria(
        datos_segmento,
        "acquisition_channel_dashboard"
    )

    visible_inicial = indice_opcion == 0

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",
            value=kpis_segmento["clientes"],
            number={"valueformat": ",d"},
            title={"text": "Clientes"},
            visible=visible_inicial
        ),
        row=1,
        col=1
    )

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",
            value=kpis_segmento["tasa_churn_pct"],
            number={
                "valueformat": ".2f",
                "suffix": " %"
            },
            title={"text": "Tasa de churn"},
            visible=visible_inicial
        ),
        row=1,
        col=2
    )

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",
            value=(
                kpis_segmento["gasto_promedio"]
                if kpis_segmento["gasto_promedio"]
                is not None
                else 0
            ),
            number={
                "valueformat": ",.2f"
            },
            title={"text": "Gasto promedio"},
            visible=visible_inicial
        ),
        row=1,
        col=3
    )

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",
            value=(
                kpis_segmento["satisfaccion_promedio"]
                if kpis_segmento["satisfaccion_promedio"]
                is not None
                else 0
            ),
            number={"valueformat": ".2f"},
            title={"text": "Satisfacción promedio"},
            visible=visible_inicial
        ),
        row=1,
        col=4
    )

    fig_dashboard.add_trace(
        go.Bar(
            x=tabla_paises_segmento[
                "country_dashboard"
            ],
            y=tabla_paises_segmento[
                "tasa_churn_pct"
            ],
            customdata=np.column_stack([
                tabla_paises_segmento["clientes"],
                tabla_paises_segmento["clientes_churn"]
            ]),
            text=tabla_paises_segmento[
                "tasa_churn_pct"
            ].map(lambda valor: f"{valor:.2f} %"),
            textposition="auto",
            hovertemplate=(
                "País: %{x}<br>"
                "Tasa de churn: %{y:.2f} %<br>"
                "Clientes: %{customdata[0]:,.0f}<br>"
                "Clientes churn: %{customdata[1]:,.0f}"
                "<extra></extra>"
            ),
            name="Churn por país",
            showlegend=False,
            visible=visible_inicial
        ),
        row=2,
        col=1
    )

    fig_dashboard.add_trace(
        go.Bar(
            x=tabla_canales_segmento[
                "acquisition_channel_dashboard"
            ],
            y=tabla_canales_segmento[
                "tasa_churn_pct"
            ],
            customdata=np.column_stack([
                tabla_canales_segmento["clientes"],
                tabla_canales_segmento["clientes_churn"]
            ]),
            text=tabla_canales_segmento[
                "tasa_churn_pct"
            ].map(lambda valor: f"{valor:.2f} %"),
            textposition="auto",
            hovertemplate=(
                "Canal: %{x}<br>"
                "Tasa de churn: %{y:.2f} %<br>"
                "Clientes: %{customdata[0]:,.0f}<br>"
                "Clientes churn: %{customdata[1]:,.0f}"
                "<extra></extra>"
            ),
            name="Churn por canal",
            showlegend=False,
            visible=visible_inicial
        ),
        row=2,
        col=3
    )

    fig_dashboard.add_trace(
        go.Bar(
            x=tabla_modelos_dashboard["modelo"],
            y=tabla_modelos_dashboard[
                "accuracy_prueba_pct"
            ],
            text=tabla_modelos_dashboard[
                "accuracy_prueba_pct"
            ].map(lambda valor: f"{valor:.2f} %"),
            textposition="auto",
            name="Accuracy de prueba",
            hovertemplate=(
                "Modelo: %{x}<br>"
                "Accuracy de prueba: %{y:.2f} %"
                "<extra></extra>"
            ),
            visible=visible_inicial
        ),
        row=3,
        col=1
    )

    fig_dashboard.add_trace(
        go.Bar(
            x=tabla_modelos_dashboard["modelo"],
            y=tabla_modelos_dashboard[
                "churn_detectado_pct"
            ],
            customdata=np.column_stack([
                tabla_modelos_dashboard[
                    "churn_detectados"
                ],
                tabla_modelos_dashboard[
                    "churn_omitidos"
                ]
            ]),
            text=tabla_modelos_dashboard[
                "churn_detectado_pct"
            ].map(lambda valor: f"{valor:.2f} %"),
            textposition="auto",
            name="Churn detectado",
            hovertemplate=(
                "Modelo: %{x}<br>"
                "Churn detectado: %{y:.2f} %<br>"
                "Detectados: %{customdata[0]:,.0f}<br>"
                "Omitidos: %{customdata[1]:,.0f}"
                "<extra></extra>"
            ),
            visible=visible_inicial
        ),
        row=3,
        col=1
    )

botones_pais = []
total_trazas = len(fig_dashboard.data)

for indice_opcion, opcion_pais in enumerate(opciones_pais):
    visibilidad = [False] * total_trazas
    inicio = indice_opcion * trazas_por_opcion

    for indice_traza in range(
        inicio,
        inicio + trazas_por_opcion
    ):
        visibilidad[indice_traza] = True

    botones_pais.append({
        "label": opcion_pais,
        "method": "update",
        "args": [
            {"visible": visibilidad},
            {
                "title": (
                    "Predicción de abandono de clientes | "
                    f"Segmento: {opcion_pais}"
                )
            }
        ]
    })

fig_dashboard.update_layout(
    title=(
        "Predicción de abandono de clientes | "
        "Segmento: Todos los países"
    ),
    height=1050,
    template="plotly_white",
    barmode="group",
    hovermode="closest",
    margin={
        "l": 70,
        "r": 40,
        "t": 150,
        "b": 80
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": -0.16,
        "xanchor": "center",
        "x": 0.5
    },
    updatemenus=[
        {
            "buttons": botones_pais,
            "direction": "down",
            "showactive": True,
            "x": 0.0,
            "xanchor": "left",
            "y": 1.13,
            "yanchor": "top"
        }
    ],
    annotations=[
        *list(fig_dashboard.layout.annotations),
        {
            "text": "Filtrar análisis descriptivo por país:",
            "xref": "paper",
            "yref": "paper",
            "x": 0.0,
            "y": 1.17,
            "showarrow": False,
            "xanchor": "left"
        },
        {
            "text": (
                "Los resultados del modelo corresponden al conjunto "
                "de prueba global y no cambian con el selector."
            ),
            "xref": "paper",
            "yref": "paper",
            "x": 0.5,
            "y": -0.11,
            "showarrow": False,
            "xanchor": "center"
        }
    ]
)

fig_dashboard.update_yaxes(
    ticksuffix=" %",
    rangemode="tozero",
    row=2,
    col=1
)

fig_dashboard.update_yaxes(
    ticksuffix=" %",
    rangemode="tozero",
    row=2,
    col=3
)

fig_dashboard.update_yaxes(
    ticksuffix=" %",
    range=[0, 100],
    row=3,
    col=1
)

fig_dashboard.update_xaxes(
    tickangle=-25,
    row=2,
    col=1
)

fig_dashboard.update_xaxes(
    tickangle=-25,
    row=2,
    col=3
)

fig_dashboard.update_xaxes(
    tickangle=-15,
    row=3,
    col=1
)

fig_dashboard.show(
    config={
        "responsive": True,
        "displaylogo": False
    }
)

**Interpretación.** Los indicadores resumen el tamaño y comportamiento general del segmento elegido. Las tasas por país y canal permiten comparar grupos con tamaños distintos, mientras que el panel de modelos confronta accuracy de prueba y proporción de churn detectado.

**Utilidad.** El selector facilita revisar si los patrones descriptivos generales se mantienen en cada país sin alterar el modelo entrenado.

**Limitación.** El HTML permite cambiar el país, utilizar hover, zoom y controles de Plotly, pero no implementa filtrado cruzado dinámico entre todos los gráficos. Además, una tasa alta en un grupo pequeño debe interpretarse junto con su cantidad de clientes.

### 23.4 Resumen calculado de hallazgos

Antes de exportar el archivo, se presentan los valores que sostienen la lectura principal del dashboard. Las conclusiones se derivan de las tablas calculadas durante la ejecución.

In [57]:
pais_mayor_churn = (
    tabla_paises_global
    .sort_values(
        ["tasa_churn_pct", "clientes"],
        ascending=[False, False]
    )
    .iloc[0]
)

canal_mayor_churn = (
    tabla_canales_global
    .sort_values(
        ["tasa_churn_pct", "clientes"],
        ascending=[False, False]
    )
    .iloc[0]
)

print(
    "Clientes totales:",
    f"{kpis_globales['clientes']:,}"
)

print(
    "Tasa global de churn:",
    f"{kpis_globales['tasa_churn_pct']:.2f} %"
)

print(
    "País con mayor tasa observada:",
    pais_mayor_churn["country_dashboard"],
    f"({pais_mayor_churn['tasa_churn_pct']:.2f} %, "
    f"{int(pais_mayor_churn['clientes'])} clientes)"
)

print(
    "Canal con mayor tasa observada:",
    canal_mayor_churn[
        "acquisition_channel_dashboard"
    ],
    f"({canal_mayor_churn['tasa_churn_pct']:.2f} %, "
    f"{int(canal_mayor_churn['clientes'])} clientes)"
)

if modelo_seleccionado is None:
    print(
        "Modelo seleccionado: no existe un candidato "
        "definitivo con los criterios vigentes."
    )
else:
    resumen_modelo_seleccionado = (
        tabla_modelos_dashboard.loc[
            tabla_modelos_dashboard["modelo"]
            == modelo_seleccionado
        ]
        .iloc[0]
    )

    print(
        "Modelo candidato seleccionado:",
        modelo_seleccionado
    )

    print(
        "Accuracy de prueba del candidato:",
        (
            f"{resumen_modelo_seleccionado['accuracy_prueba_pct']:.2f} %"
        )
    )

    print(
        "Churn detectado por el candidato:",
        (
            f"{resumen_modelo_seleccionado['churn_detectado_pct']:.2f} %"
        )
    )

Clientes totales: 15,000
Tasa global de churn: 15.32 %
País con mayor tasa observada: India (15.93 %, 3014 clientes)
Canal con mayor tasa observada: Organic (15.88 %, 3055 clientes)
Modelo candidato seleccionado: Regresión logística
Accuracy de prueba del candidato: 84.80 %
Churn detectado por el candidato: 19.35 %


**Interpretación.** El país y el canal destacados representan las mayores tasas observadas en el dataset, no efectos causales. Su tamaño de muestra debe revisarse antes de orientar una intervención.

La lectura del modelo debe mantener conjuntamente la accuracy y la detección de churn. Una accuracy cercana al baseline no se convierte en un resultado fuerte solo por presentar estabilidad entre entrenamiento y prueba.

**Recomendación.** Utilizar el dashboard como apoyo descriptivo para priorizar análisis de segmentos, no como un sistema automático de decisión individual. Cualquier acción de retención debe considerar costos, capacidad operativa y validación posterior.

### 23.5 Exportación del archivo HTML

La figura se guarda como un archivo HTML autónomo con la librería de Plotly incorporada. De esta forma puede abrirse localmente incluso cuando no exista conexión a Internet.

In [58]:
carpeta_dashboard = (
    ruta_raiz_proyecto
    / "dashboard"
)

carpeta_dashboard.mkdir(
    parents=True,
    exist_ok=True
)

ruta_dashboard_html = (
    carpeta_dashboard
    / "dashboard_churn.html"
)

fig_dashboard.write_html(
    ruta_dashboard_html,
    include_plotlyjs=True,
    full_html=True,
    auto_open=False,
    config={
        "responsive": True,
        "displaylogo": False
    }
)

if not ruta_dashboard_html.exists():
    raise FileNotFoundError(
        "No se generó dashboard_churn.html."
    )

if ruta_dashboard_html.stat().st_size == 0:
    raise ValueError(
        "El archivo HTML generado está vacío."
    )

print("Archivo HTML interactivo exportado correctamente.")
print("Ruta:", ruta_dashboard_html)
print(
    "Tamaño del archivo:",
    f"{ruta_dashboard_html.stat().st_size / 1024 / 1024:.2f} MB"
)
print(
    "Opciones del selector:",
    len(opciones_pais)
)

Archivo HTML interactivo exportado correctamente.
Ruta: ..\dashboard\dashboard_churn.html
Tamaño del archivo: 4.66 MB
Opciones del selector: 6


### 23.6 Conclusión y limitaciones del dashboard

El dashboard consolida cuatro indicadores y tres vistas complementarias en un archivo HTML reproducible. El selector de país actualiza los indicadores y las comparaciones descriptivas, mientras que el rendimiento de los modelos se conserva como resultado global del conjunto de prueba.

La exportación incorpora Plotly dentro del HTML, por lo que el archivo puede abrirse sin ejecutar Python y funciona como contingencia sin conexión. No requiere servidor, callbacks ni una aplicación web adicional.

Limitaciones:

- no existe filtrado cruzado entre todos los gráficos;
- el selector no reentrena ni reevalúa modelos por país;
- las comparaciones segmentadas son descriptivas y no causales;
- los promedios pueden ocultar dispersión y valores faltantes;
- los indicadores externos representan contexto nacional repetido entre clientes;
- la utilidad del modelo continúa limitada si su mejora frente al baseline y su detección de churn son reducidas;
- el HTML debe regenerarse cuando cambie el dataset procesado o los resultados del modelado.